In [ ]:
# ============================================================
# STEP 0: CONNECT GOOGLE DRIVE + CREATE A NEW PROJECT FOLDER
# Everything generated in this notebook will be saved here.
# ============================================================

from google.colab import drive, files
import os

drive.mount("/content/drive")

# CHANGE ONLY THIS NAME if you want another project folder name
PROJECT_DIR = "/content/drive/MyDrive/Phishing_Detection_Project_New"

# Create organized subfolders
RAW_DIR = f"{PROJECT_DIR}/01_Raw_Data"
CLEAN_DIR = f"{PROJECT_DIR}/02_Cleaned_Data"
FEATURE_DIR = f"{PROJECT_DIR}/03_Feature_Selection"
SPLIT_DIR = f"{PROJECT_DIR}/04_Train_Test_Data"
MODEL_DIR = f"{PROJECT_DIR}/05_Saved_Models"
RESULT_DIR = f"{PROJECT_DIR}/06_Results"
XAI_DIR = f"{PROJECT_DIR}/07_XAI"

for folder in [PROJECT_DIR, RAW_DIR, CLEAN_DIR, FEATURE_DIR, SPLIT_DIR, MODEL_DIR, RESULT_DIR, XAI_DIR]:
    os.makedirs(folder, exist_ok=True)

print("Project folder ready:")
print(PROJECT_DIR)

# Upload the ORIGINAL RAW CSV dataset from your computer
print("\nPlease upload the ORIGINAL RAW dataset CSV file")
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
raw_dataset_path = f"{RAW_DIR}/{file_name}"

# Save uploaded raw dataset permanently to the new Google Drive folder
with open(raw_dataset_path, "wb") as f:
    f.write(uploaded[file_name])

print("\nRaw dataset saved successfully!")
print(raw_dataset_path)

Mounted at /content/drive
Project folder ready:
/content/drive/MyDrive/Phishing_Detection_Project_New

Please upload the ORIGINAL RAW dataset CSV file


Saving dataset_full.csv to dataset_full.csv

Raw dataset saved successfully!
/content/drive/MyDrive/Phishing_Detection_Project_New/01_Raw_Data/dataset_full.csv


## STEP 1 — Import Libraries and Load Raw Dataset


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# ============================================================
# STEP 1: LOAD ORIGINAL RAW DATASET
# The raw dataset is loaded from the NEW project folder.
# ============================================================

import pandas as pd
df = pd.read_csv(raw_dataset_path)
print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
df.head()

Dataset loaded successfully!
Dataset shape: (88647, 112)


,qty_dot_url,qty_hyphen_url,qty_underline_url,qty_slash_url,qty_questionmark_url,qty_equal_url,qty_at_url,qty_and_url,qty_exclamation_url,qty_space_url,...,qty_ip_resolved,qty_nameservers,qty_mx_servers,ttl_hostname,tls_ssl_certificate,qty_redirects,url_google_index,domain_google_index,url_shortened,phishing
0,3,0,0,1,0,0,0,0,0,0,...,1,2,0,892,0,0,0,0,0,1
1,5,0,1,3,0,3,0,2,0,0,...,1,2,1,9540,1,0,0,0,0,1
2,2,0,0,1,0,0,0,0,0,0,...,1,2,3,589,1,0,0,0,0,0
3,4,0,2,5,0,0,0,0,0,0,...,1,2,0,292,1,0,0,0,0,1
4,2,0,0,0,0,0,0,0,0,0,...,1,2,1,3597,0,1,0,0,0,0


## STEP 2 — Initial Dataset Inspection


In [ ]:
#row cloumn dekha
print(df.head())

   qty_dot_url  qty_hyphen_url  qty_underline_url  qty_slash_url  \
0            3               0                  0              1   
1            5               0                  1              3   
2            2               0                  0              1   
3            4               0                  2              5   
4            2               0                  0              0   

   qty_questionmark_url  qty_equal_url  qty_at_url  qty_and_url  \
0                     0              0           0            0   
1                     0              3           0            2   
2                     0              0           0            0   
3                     0              0           0            0   
4                     0              0           0            0   

   qty_exclamation_url  qty_space_url  ...  qty_ip_resolved  qty_nameservers  \
0                    0              0  ...                1                2   
1                    0      

In [ ]:
#column name dekha
print("\nColumn names:")
print(df.columns.tolist())


Column names:
['qty_dot_url', 'qty_hyphen_url', 'qty_underline_url', 'qty_slash_url', 'qty_questionmark_url', 'qty_equal_url', 'qty_at_url', 'qty_and_url', 'qty_exclamation_url', 'qty_space_url', 'qty_tilde_url', 'qty_comma_url', 'qty_plus_url', 'qty_asterisk_url', 'qty_hashtag_url', 'qty_dollar_url', 'qty_percent_url', 'qty_tld_url', 'length_url', 'qty_dot_domain', 'qty_hyphen_domain', 'qty_underline_domain', 'qty_slash_domain', 'qty_questionmark_domain', 'qty_equal_domain', 'qty_at_domain', 'qty_and_domain', 'qty_exclamation_domain', 'qty_space_domain', 'qty_tilde_domain', 'qty_comma_domain', 'qty_plus_domain', 'qty_asterisk_domain', 'qty_hashtag_domain', 'qty_dollar_domain', 'qty_percent_domain', 'qty_vowels_domain', 'domain_length', 'domain_in_ip', 'server_client_domain', 'qty_dot_directory', 'qty_hyphen_directory', 'qty_underline_directory', 'qty_slash_directory', 'qty_questionmark_directory', 'qty_equal_directory', 'qty_at_directory', 'qty_and_directory', 'qty_exclamation_direct

In [ ]:
#data type dekha
print("\nData types:")
print(df.dtypes.value_counts())


Data types:
int64      111
float64      1
Name: count, dtype: int64


In [ ]:
#target column er values
print("\nTarget column:")
print(df["phishing"].value_counts())


Target column:
phishing
0    58000
1    30647
Name: count, dtype: int64


In [ ]:
#ratio target column er
class_percentage = (
    df["phishing"]
    .value_counts(normalize=True)
    * 100
)

print("Class percentage:")
print(class_percentage)

Class percentage:
phishing
0    65.428046
1    34.571954
Name: proportion, dtype: float64


In [ ]:
missing_values = df.isnull().sum()

print("Total missing values:", missing_values.sum())

Total missing values: 0


In [ ]:
duplicate_rows = df.duplicated().sum()

print("Duplicate rows:", duplicate_rows)

Duplicate rows: 1438


## STEP 3 — Find and Remove Conflicting Labels
Same feature values with different phishing labels are removed first.


In [ ]:
# Target column বাদ দিয়ে feature columns নেওয়া
feature_cols = [
    col for col in df.columns
    if col != "phishing"
]

# একই feature values আছে এমন rows খুঁজে বের করা
duplicate_features = df[
    df.duplicated(
        subset=feature_cols,
        keep=False
    )
]

print(
    "Rows with duplicate feature values:",
    len(duplicate_features)
)

Rows with duplicate feature values: 1937


In [ ]:
# একই feature values-এর জন্য কয় ধরনের label আছে তা দেখা
label_conflicts = (
    duplicate_features
    .groupby(feature_cols)["phishing"]
    .nunique()
)

print(
    "Conflicting duplicate feature groups:",
    (label_conflicts > 1).sum()
)

Conflicting duplicate feature groups: 6


In [ ]:
# Find feature groups where the same features have different labels

conflicting_groups = label_conflicts[
    label_conflicts > 1
]

print("Number of conflicting groups:",
      len(conflicting_groups))

Number of conflicting groups: 6


In [ ]:
# Count total rows belonging to conflicting groups

conflicting_rows = duplicate_features[
    duplicate_features.set_index(feature_cols).index.isin(
        conflicting_groups.index
    )
]

print("Total conflicting rows:",
      len(conflicting_rows))

Total conflicting rows: 13


In [ ]:
print("\nLabel distribution among conflicting rows:")
print(conflicting_rows["phishing"].value_counts())



Label distribution among conflicting rows:
phishing
0    7
1    6
Name: count, dtype: int64


In [ ]:
# Remove the conflicting rows
df_clean = df.drop(index=conflicting_rows.index)

# Reset the row index
df_clean = df_clean.reset_index(drop=True)

print("Original dataset shape:", df.shape)
print("After removing conflicting rows:", df_clean.shape)
print("Rows removed:", len(df) - len(df_clean))

Original dataset shape: (88647, 112)
After removing conflicting rows: (88634, 112)
Rows removed: 13


## STEP 4 — Remove Exact Duplicate Rows


In [ ]:
duplicate_clean = df_clean.duplicated().sum()

print("Exact duplicate rows after conflict removal:",
      duplicate_clean)

Exact duplicate rows after conflict removal: 1437


In [ ]:
# Remove exact duplicate rows
df_clean = df_clean.drop_duplicates()

# Reset index
df_clean = df_clean.reset_index(drop=True)

print("Dataset shape after removing exact duplicates:",
      df_clean.shape)

print("Rows remaining:", len(df_clean))

Dataset shape after removing exact duplicates: (87197, 112)
Rows remaining: 87197


In [ ]:
remaining_duplicates = df_clean.duplicated().sum()

print("Remaining duplicate rows:", remaining_duplicates)

Remaining duplicate rows: 0


In [ ]:
print("Cleaned class distribution:")
print(df_clean["phishing"].value_counts())

print("\nCleaned class percentage:")
print(
    df_clean["phishing"]
    .value_counts(normalize=True)
    .mul(100)
)

Cleaned class distribution:
phishing
0    56706
1    30491
Name: count, dtype: int64

Cleaned class percentage:
phishing
0    65.032054
1    34.967946
Name: proportion, dtype: float64


## STEP 5 — Remove Constant Features and Save Clean Dataset


In [ ]:
# Check the number of unique values in each feature
unique_counts = df_clean.drop(columns=["phishing"]).nunique()

# Find features with only one unique value
constant_features = unique_counts[
    unique_counts == 1
]

print("Number of constant features:",
      len(constant_features))

print("\nConstant features:")
print(constant_features)

Number of constant features: 13

Constant features:
qty_slash_domain           1
qty_questionmark_domain    1
qty_equal_domain           1
qty_and_domain             1
qty_exclamation_domain     1
qty_space_domain           1
qty_tilde_domain           1
qty_comma_domain           1
qty_plus_domain            1
qty_asterisk_domain        1
qty_hashtag_domain         1
qty_dollar_domain          1
qty_percent_domain         1
dtype: int64


In [ ]:
# Remove constant features
df_clean = df_clean.drop(
    columns=constant_features.index
)

print("Original number of features: 111")
print("Removed constant features:", len(constant_features))
print("Remaining features:",
      df_clean.shape[1] - 1)  # -1 for target


Original number of features: 111
Removed constant features: 13
Remaining features: 98


In [ ]:
clean_path = f"{CLEAN_DIR}/cleaned_dataset.csv"

df_clean.to_csv(clean_path, index=False)

print("Cleaned dataset saved successfully!")
print(clean_path)

Cleaned dataset saved successfully!
/content/drive/MyDrive/Phishing_Detection_Project_New/02_Cleaned_Data/cleaned_dataset.csv


## STEP 6 — Random Forest Feature Importance
Rank the cleaned features and select approximately 95% cumulative importance (47 features).


In [ ]:
# Select only input features
X = df_clean.drop(columns=["phishing"])

# Calculate feature-to-feature correlation
correlation_matrix = X.corr()

print("Correlation matrix shape:",
      correlation_matrix.shape)

Correlation matrix shape: (98, 98)


In [ ]:
# Find highly correlated feature pairs
high_corr_pairs = []

for i in range(len(correlation_matrix.columns)):
    for j in range(i + 1, len(correlation_matrix.columns)):

        corr_value = correlation_matrix.iloc[i, j]

        if abs(corr_value) >= 0.90:
            high_corr_pairs.append((
                correlation_matrix.columns[i],
                correlation_matrix.columns[j],
                corr_value
            ))

print("Highly correlated pairs:",
      len(high_corr_pairs))

print("\nFirst 20 highly correlated pairs:")

for pair in high_corr_pairs[:20]:
    print(pair)

Highly correlated pairs: 337

First 20 highly correlated pairs:
('qty_slash_url', 'qty_slash_directory', np.float64(0.963254040421188))
('qty_equal_url', 'qty_equal_params', np.float64(0.9500149726961067))
('qty_equal_url', 'qty_and_params', np.float64(0.9235252417496727))
('qty_and_url', 'qty_and_params', np.float64(0.9142957441884971))
('qty_dot_directory', 'qty_dot_file', np.float64(0.9151189983301599))
('qty_underline_directory', 'qty_underline_file', np.float64(0.9341514629708438))
('qty_questionmark_directory', 'qty_equal_directory', np.float64(0.9791493837580579))
('qty_questionmark_directory', 'qty_at_directory', np.float64(0.9092170365952988))
('qty_questionmark_directory', 'qty_and_directory', np.float64(0.9346569531580747))
('qty_questionmark_directory', 'qty_exclamation_directory', np.float64(0.9927739322242058))
('qty_questionmark_directory', 'qty_space_directory', np.float64(0.9911268178570188))
('qty_questionmark_directory', 'qty_tilde_directory', np.float64(0.9880641986

In [ ]:
# Find features that are involved in high-correlation pairs

redundant_candidates = set()

for feature1, feature2, corr_value in high_corr_pairs:
    redundant_candidates.add(feature2)

print("Number of candidate redundant features:",
      len(redundant_candidates))

print("\nCandidate redundant features:")

for feature in sorted(redundant_candidates):
    print(feature)

Number of candidate redundant features: 41

Candidate redundant features:
qty_and_directory
qty_and_file
qty_and_params
qty_asterisk_directory
qty_asterisk_file
qty_asterisk_params
qty_at_directory
qty_at_file
qty_at_params
qty_comma_directory
qty_comma_file
qty_comma_params
qty_dollar_directory
qty_dollar_file
qty_dollar_params
qty_dot_file
qty_equal_directory
qty_equal_file
qty_equal_params
qty_exclamation_directory
qty_exclamation_file
qty_exclamation_params
qty_hashtag_directory
qty_hashtag_file
qty_hashtag_params
qty_params
qty_percent_file
qty_plus_directory
qty_plus_file
qty_plus_params
qty_questionmark_file
qty_slash_directory
qty_slash_file
qty_space_directory
qty_space_file
qty_space_params
qty_tilde_directory
qty_tilde_file
qty_tilde_params
qty_underline_file
tld_present_params


In [ ]:
# Calculate correlation of each candidate feature with the target

target_correlation = (
    df_clean[list(redundant_candidates) + ["phishing"]]
    .corr()["phishing"]
    .drop("phishing")
    .abs()
    .sort_values(ascending=False)
)

print("Candidate features ranked by correlation with target:")
print(target_correlation)

Candidate features ranked by correlation with target:
qty_slash_directory          0.744759
qty_hashtag_file             0.744656
qty_questionmark_file        0.744656
qty_slash_file               0.744656
qty_hashtag_directory        0.744656
qty_dollar_file              0.744656
qty_at_file                  0.744518
qty_exclamation_file         0.744059
qty_and_file                 0.743856
qty_tilde_file               0.743655
qty_equal_file               0.742671
qty_comma_file               0.742358
qty_comma_directory          0.742241
qty_space_file               0.741720
qty_exclamation_directory    0.741145
qty_space_directory          0.738958
qty_tilde_directory          0.738489
qty_equal_directory          0.736901
qty_plus_file                0.734007
qty_dollar_directory         0.733785
qty_dot_file                 0.732459
qty_plus_directory           0.731678
qty_and_directory            0.700723
qty_asterisk_file            0.683072
qty_at_directory             0.680

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Input features and target
X = df_clean.drop(columns=["phishing"])
y = df_clean["phishing"]

# Random Forest model
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Train the model
rf.fit(X, y)

print("Random Forest training completed.")

Random Forest training completed.


In [ ]:
# Get feature importance
feature_importance = pd.Series(
    rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("Top 30 features based on Random Forest importance:")
print(feature_importance.head(30))

Top 30 features based on Random Forest importance:
directory_length             0.087248
time_domain_activation       0.063240
qty_hyphen_directory         0.032549
length_url                   0.032079
file_length                  0.028637
qty_slash_directory          0.028540
qty_at_file                  0.028142
qty_slash_file               0.027716
qty_dot_directory            0.027207
qty_dot_file                 0.026546
ttl_hostname                 0.025937
asn_ip                       0.024963
time_response                0.023292
qty_percent_file             0.022838
qty_equal_directory          0.022810
qty_space_file               0.022350
qty_asterisk_file            0.022245
qty_exclamation_directory    0.022156
time_domain_expiration       0.022040
domain_length                0.018860
qty_hyphen_file              0.018775
qty_underline_directory      0.018076
qty_slash_url                0.017151
qty_tilde_file               0.017063
qty_asterisk_directory       0.017053

In [ ]:
# Sort feature importance
importance_df = (
    feature_importance
    .reset_index()
)

importance_df.columns = [
    "Feature",
    "Importance"
]

# Calculate cumulative importance
importance_df["Cumulative_Importance"] = (
    importance_df["Importance"].cumsum()
)

print(importance_df.head(30))

                      Feature  Importance  Cumulative_Importance
0            directory_length    0.087248               0.087248
1      time_domain_activation    0.063240               0.150488
2        qty_hyphen_directory    0.032549               0.183037
3                  length_url    0.032079               0.215116
4                 file_length    0.028637               0.243752
5         qty_slash_directory    0.028540               0.272293
6                 qty_at_file    0.028142               0.300435
7              qty_slash_file    0.027716               0.328151
8           qty_dot_directory    0.027207               0.355358
9                qty_dot_file    0.026546               0.381904
10               ttl_hostname    0.025937               0.407841
11                     asn_ip    0.024963               0.432803
12              time_response    0.023292               0.456095
13           qty_percent_file    0.022838               0.478933
14        qty_equal_direc

In [ ]:
# Find number of features needed for different
# cumulative importance thresholds

for threshold in [0.80, 0.90, 0.95]:

    number_of_features = (
        importance_df["Cumulative_Importance"] >= threshold
    ).idxmax() + 1

    print(
        f"Features needed for approximately "
        f"{threshold*100:.0f}% cumulative importance:",
        number_of_features
    )

Features needed for approximately 80% cumulative importance: 32
Features needed for approximately 90% cumulative importance: 40
Features needed for approximately 95% cumulative importance: 47


In [ ]:
# Select features covering approximately 95% cumulative importance

selected_features = importance_df[
    importance_df["Cumulative_Importance"] <= 0.95
]["Feature"].tolist()

# Make sure the first feature crossing 95% is also included
if len(selected_features) < 47:
    selected_features.append(
        importance_df.iloc[len(selected_features)]["Feature"]
    )

print("Number of features selected:",
      len(selected_features))

print("\nSelected features:")
print(selected_features)

Number of features selected: 47

Selected features:
['directory_length', 'time_domain_activation', 'qty_hyphen_directory', 'length_url', 'file_length', 'qty_slash_directory', 'qty_at_file', 'qty_slash_file', 'qty_dot_directory', 'qty_dot_file', 'ttl_hostname', 'asn_ip', 'time_response', 'qty_percent_file', 'qty_equal_directory', 'qty_space_file', 'qty_asterisk_file', 'qty_exclamation_directory', 'time_domain_expiration', 'domain_length', 'qty_hyphen_file', 'qty_underline_directory', 'qty_slash_url', 'qty_tilde_file', 'qty_asterisk_directory', 'qty_equal_file', 'qty_plus_directory', 'qty_dollar_directory', 'qty_dollar_file', 'qty_space_directory', 'qty_dot_domain', 'qty_vowels_domain', 'qty_nameservers', 'qty_mx_servers', 'qty_and_file', 'qty_comma_directory', 'qty_plus_file', 'qty_hashtag_directory', 'qty_and_directory', 'qty_comma_file', 'qty_dot_url', 'qty_ip_resolved', 'qty_redirects', 'qty_hyphen_url', 'qty_questionmark_directory', 'tls_ssl_certificate', 'qty_at_directory']


## STEP 7 — Correlation-Based Reduction: 47 → 27 Features
Highly correlated features are grouped, and representative features are retained.


In [ ]:
# Correlation matrix for the selected 47 features

selected_corr_matrix = df_clean[selected_features].corr()

# Find highly correlated pairs among selected features
high_corr_selected = []

for i in range(len(selected_corr_matrix.columns)):
    for j in range(i + 1, len(selected_corr_matrix.columns)):

        corr_value = selected_corr_matrix.iloc[i, j]

        if abs(corr_value) >= 0.90:
            high_corr_selected.append((
                selected_corr_matrix.columns[i],
                selected_corr_matrix.columns[j],
                corr_value
            ))

print("Highly correlated pairs among selected features:",
      len(high_corr_selected))

print("\nFirst 20 highly correlated pairs:")

for pair in high_corr_selected[:20]:
    print(pair)

Highly correlated pairs among selected features: 189

First 20 highly correlated pairs:
('qty_slash_directory', 'qty_slash_url', np.float64(0.963254040421188))
('qty_at_file', 'qty_slash_file', np.float64(0.9993570859264459))
('qty_at_file', 'qty_equal_directory', np.float64(0.9785022420865057))
('qty_at_file', 'qty_space_file', np.float64(0.9947491664046557))
('qty_at_file', 'qty_asterisk_file', np.float64(0.9152847147007979))
('qty_at_file', 'qty_exclamation_directory', np.float64(0.9921769974266849))
('qty_at_file', 'qty_tilde_file', np.float64(0.9980721843534949))
('qty_at_file', 'qty_equal_file', np.float64(0.9943306405192225))
('qty_at_file', 'qty_plus_directory', np.float64(0.983723456938829))
('qty_at_file', 'qty_dollar_directory', np.float64(0.9827601770259642))
('qty_at_file', 'qty_dollar_file', np.float64(0.9993570859264459))
('qty_at_file', 'qty_space_directory', np.float64(0.9904875823569554))
('qty_at_file', 'qty_and_file', np.float64(0.996449161645172))
('qty_at_file', '

In [ ]:
# Find unique candidate redundant features
redundant_selected = set()

for feature1, feature2, corr_value in high_corr_selected:
    redundant_selected.add(feature2)

print("Number of redundant candidates:",
      len(redundant_selected))

print("\nRedundant candidate features:")

for feature in sorted(redundant_selected):
    print(feature)

Number of redundant candidates: 22

Redundant candidate features:
qty_and_directory
qty_and_file
qty_asterisk_directory
qty_asterisk_file
qty_at_directory
qty_comma_directory
qty_comma_file
qty_dollar_directory
qty_dollar_file
qty_dot_file
qty_equal_directory
qty_equal_file
qty_exclamation_directory
qty_hashtag_directory
qty_plus_directory
qty_plus_file
qty_questionmark_directory
qty_slash_file
qty_slash_url
qty_space_directory
qty_space_file
qty_tilde_file


In [ ]:
# Create correlation groups among selected features

correlation_groups = []

visited = set()

for feature in selected_features:

    if feature in visited:
        continue

    group = set([feature])

    for feature1, feature2, corr_value in high_corr_selected:

        if abs(corr_value) >= 0.90:

            if feature1 == feature:
                group.add(feature2)

            elif feature2 == feature:
                group.add(feature1)

    if len(group) > 1:
        correlation_groups.append(group)
        visited.update(group)

print("Number of correlation groups:",
      len(correlation_groups))

for i, group in enumerate(correlation_groups, 1):
    print(f"\nGroup {i}:")
    print(sorted(group))

Number of correlation groups: 4

Group 1:
['qty_slash_directory', 'qty_slash_url']

Group 2:
['qty_and_directory', 'qty_and_file', 'qty_asterisk_file', 'qty_at_directory', 'qty_at_file', 'qty_comma_directory', 'qty_comma_file', 'qty_dollar_directory', 'qty_dollar_file', 'qty_equal_directory', 'qty_equal_file', 'qty_exclamation_directory', 'qty_hashtag_directory', 'qty_plus_directory', 'qty_plus_file', 'qty_questionmark_directory', 'qty_slash_file', 'qty_space_directory', 'qty_space_file', 'qty_tilde_file']

Group 3:
['qty_dot_directory', 'qty_dot_file']

Group 4:
['qty_and_directory', 'qty_asterisk_directory', 'qty_asterisk_file', 'qty_dollar_directory']


In [ ]:
import networkx as nx

# Create graph
G = nx.Graph()

# Add all selected features as nodes
G.add_nodes_from(selected_features)

# Add edges for highly correlated pairs
for feature1, feature2, corr_value in high_corr_selected:

    if abs(corr_value) >= 0.90:
        G.add_edge(feature1, feature2)

# Find connected components
correlation_clusters = list(nx.connected_components(G))

# Keep only clusters containing more than one feature
correlation_clusters = [
    cluster for cluster in correlation_clusters
    if len(cluster) > 1
]

print("Number of correlation clusters:",
      len(correlation_clusters))

for i, cluster in enumerate(correlation_clusters, 1):
    print(f"\nCluster {i} ({len(cluster)} features):")
    print(sorted(cluster))

Number of correlation clusters: 3

Cluster 1 (2 features):
['qty_slash_directory', 'qty_slash_url']

Cluster 2 (21 features):
['qty_and_directory', 'qty_and_file', 'qty_asterisk_directory', 'qty_asterisk_file', 'qty_at_directory', 'qty_at_file', 'qty_comma_directory', 'qty_comma_file', 'qty_dollar_directory', 'qty_dollar_file', 'qty_equal_directory', 'qty_equal_file', 'qty_exclamation_directory', 'qty_hashtag_directory', 'qty_plus_directory', 'qty_plus_file', 'qty_questionmark_directory', 'qty_slash_file', 'qty_space_directory', 'qty_space_file', 'qty_tilde_file']

Cluster 3 (2 features):
['qty_dot_directory', 'qty_dot_file']


In [ ]:
# Compare Random Forest importance and target correlation
# for each correlation cluster

for i, cluster in enumerate(correlation_clusters, 1):

    cluster_df = importance_df[
        importance_df["Feature"].isin(cluster)
    ].copy()

    cluster_df["Target_Correlation"] = (
        cluster_df["Feature"]
        .map(target_correlation)
    )

    cluster_df = cluster_df.sort_values(
        by="Importance",
        ascending=False
    )

    print(f"\n{'='*60}")
    print(f"Cluster {i}")
    print(f"{'='*60}")

    print(
        cluster_df[
            ["Feature", "Importance", "Target_Correlation"]
        ].to_string(index=False)
    )


Cluster 1
            Feature  Importance  Target_Correlation
qty_slash_directory    0.028540            0.744759
      qty_slash_url    0.017151                 NaN

Cluster 2
                   Feature  Importance  Target_Correlation
               qty_at_file    0.028142            0.744518
            qty_slash_file    0.027716            0.744656
       qty_equal_directory    0.022810            0.736901
            qty_space_file    0.022350            0.741720
         qty_asterisk_file    0.022245            0.683072
 qty_exclamation_directory    0.022156            0.741145
            qty_tilde_file    0.017063            0.743655
    qty_asterisk_directory    0.017053            0.649489
            qty_equal_file    0.016772            0.742671
        qty_plus_directory    0.016766            0.731678
      qty_dollar_directory    0.016735            0.733785
           qty_dollar_file    0.016589            0.744656
       qty_space_directory    0.016580            0.738

In [ ]:
# Calculate target correlation directly for all selected features

selected_target_corr = (
    df_clean[selected_features]
    .corrwith(df_clean["phishing"])
    .abs()
    .sort_values(ascending=False)
)

print("Target correlation for selected features:")
print(selected_target_corr)

Target correlation for selected features:
qty_slash_directory           0.744759
qty_slash_file                0.744656
qty_dollar_file               0.744656
qty_hashtag_directory         0.744656
qty_questionmark_directory    0.744656
qty_at_file                   0.744518
qty_and_file                  0.743856
qty_tilde_file                0.743655
qty_equal_file                0.742671
qty_comma_file                0.742358
qty_comma_directory           0.742241
qty_space_file                0.741720
qty_exclamation_directory     0.741145
qty_space_directory           0.738958
qty_equal_directory           0.736901
qty_plus_file                 0.734007
qty_dollar_directory          0.733785
qty_dot_file                  0.732459
qty_plus_directory            0.731678
qty_and_directory             0.700723
qty_slash_url                 0.697020
qty_dot_directory             0.689257
qty_asterisk_file             0.683072
qty_at_directory              0.680521
qty_asterisk_directory

In [ ]:
selected_features = [
    'directory_length',
    'time_domain_activation',
    'qty_hyphen_directory',
    'length_url',
    'file_length',
    'qty_slash_directory',
    'qty_at_file',
    'qty_slash_file',
    'qty_dot_directory',
    'qty_dot_file',
    'ttl_hostname',
    'asn_ip',
    'time_response',
    'qty_percent_file',
    'qty_equal_directory',
    'qty_space_file',
    'qty_asterisk_file',
    'qty_exclamation_directory',
    'time_domain_expiration',
    'domain_length',
    'qty_hyphen_file',
    'qty_underline_directory',
    'qty_slash_url',
    'qty_tilde_file',
    'qty_asterisk_directory',
    'qty_equal_file',
    'qty_plus_directory',
    'qty_dollar_directory',
    'qty_dollar_file',
    'qty_space_directory',
    'qty_dot_domain',
    'qty_vowels_domain',
    'qty_nameservers',
    'qty_mx_servers',
    'qty_and_file',
    'qty_comma_directory',
    'qty_plus_file',
    'qty_hashtag_directory',
    'qty_and_directory',
    'qty_comma_file',
    'qty_dot_url',
    'qty_ip_resolved',
    'qty_redirects',
    'qty_hyphen_url',
    'qty_questionmark_directory',
    'tls_ssl_certificate',
    'qty_at_directory'
]

print("Number of selected features:", len(selected_features))

Number of selected features: 47


In [ ]:
selected_corr = df_clean[selected_features].corr()

print("Correlation matrix shape:", selected_corr.shape)

Correlation matrix shape: (47, 47)


In [ ]:
# Find highly correlated feature pairs

high_corr_selected = []

for i in range(len(selected_features)):
    for j in range(i + 1, len(selected_features)):

        feature1 = selected_features[i]
        feature2 = selected_features[j]

        corr_value = selected_corr.loc[feature1, feature2]

        if abs(corr_value) >= 0.90:
            high_corr_selected.append(
                (feature1, feature2, corr_value)
            )

print("Highly correlated pairs:",
      len(high_corr_selected))

print("\nFirst 20 highly correlated pairs:")

for pair in high_corr_selected[:20]:
    print(pair)

Highly correlated pairs: 189

First 20 highly correlated pairs:
('qty_slash_directory', 'qty_slash_url', np.float64(0.963254040421188))
('qty_at_file', 'qty_slash_file', np.float64(0.9993570859264459))
('qty_at_file', 'qty_equal_directory', np.float64(0.9785022420865057))
('qty_at_file', 'qty_space_file', np.float64(0.9947491664046557))
('qty_at_file', 'qty_asterisk_file', np.float64(0.9152847147007979))
('qty_at_file', 'qty_exclamation_directory', np.float64(0.9921769974266849))
('qty_at_file', 'qty_tilde_file', np.float64(0.9980721843534949))
('qty_at_file', 'qty_equal_file', np.float64(0.9943306405192225))
('qty_at_file', 'qty_plus_directory', np.float64(0.983723456938829))
('qty_at_file', 'qty_dollar_directory', np.float64(0.9827601770259642))
('qty_at_file', 'qty_dollar_file', np.float64(0.9993570859264459))
('qty_at_file', 'qty_space_directory', np.float64(0.9904875823569554))
('qty_at_file', 'qty_and_file', np.float64(0.996449161645172))
('qty_at_file', 'qty_comma_directory', np

In [ ]:
import networkx as nx

# Create graph
G = nx.Graph()

# Add all 47 selected features as nodes
G.add_nodes_from(selected_features)

# Add an edge when correlation >= 0.90
for feature1, feature2, corr_value in high_corr_selected:
    if abs(corr_value) >= 0.90:
        G.add_edge(feature1, feature2)

# Find connected components
correlation_clusters = list(nx.connected_components(G))

# Keep only clusters with more than one feature
correlation_clusters = [
    cluster
    for cluster in correlation_clusters
    if len(cluster) > 1
]

print("Number of correlation clusters:",
      len(correlation_clusters))

for i, cluster in enumerate(correlation_clusters, 1):
    print(f"\nCluster {i} ({len(cluster)} features):")
    print(sorted(cluster))

Number of correlation clusters: 3

Cluster 1 (2 features):
['qty_slash_directory', 'qty_slash_url']

Cluster 2 (21 features):
['qty_and_directory', 'qty_and_file', 'qty_asterisk_directory', 'qty_asterisk_file', 'qty_at_directory', 'qty_at_file', 'qty_comma_directory', 'qty_comma_file', 'qty_dollar_directory', 'qty_dollar_file', 'qty_equal_directory', 'qty_equal_file', 'qty_exclamation_directory', 'qty_hashtag_directory', 'qty_plus_directory', 'qty_plus_file', 'qty_questionmark_directory', 'qty_slash_file', 'qty_space_directory', 'qty_space_file', 'qty_tilde_file']

Cluster 3 (2 features):
['qty_dot_directory', 'qty_dot_file']


In [ ]:
# Cluster-wise comparison of Random Forest importance
# and target correlation

# Calculate target correlation for the 47 selected features
selected_target_corr = (
    df_clean[selected_features]
    .corrwith(df_clean["phishing"])
    .abs()
)

for i, cluster in enumerate(correlation_clusters, 1):

    cluster_df = importance_df[
        importance_df["Feature"].isin(cluster)
    ].copy()

    cluster_df["Target_Correlation"] = cluster_df["Feature"].map(
        selected_target_corr
    )

    cluster_df = cluster_df.sort_values(
        by="Importance",
        ascending=False
    )

    print(f"\n{'='*70}")
    print(f"CLUSTER {i} — {len(cluster)} features")
    print(f"{'='*70}")

    print(
        cluster_df[
            ["Feature", "Importance", "Target_Correlation"]
        ].to_string(index=False)
    )


CLUSTER 1 — 2 features
            Feature  Importance  Target_Correlation
qty_slash_directory    0.028540            0.744759
      qty_slash_url    0.017151            0.697020

CLUSTER 2 — 21 features
                   Feature  Importance  Target_Correlation
               qty_at_file    0.028142            0.744518
            qty_slash_file    0.027716            0.744656
       qty_equal_directory    0.022810            0.736901
            qty_space_file    0.022350            0.741720
         qty_asterisk_file    0.022245            0.683072
 qty_exclamation_directory    0.022156            0.741145
            qty_tilde_file    0.017063            0.743655
    qty_asterisk_directory    0.017053            0.649489
            qty_equal_file    0.016772            0.742671
        qty_plus_directory    0.016766            0.731678
      qty_dollar_directory    0.016735            0.733785
           qty_dollar_file    0.016589            0.744656
       qty_space_directory  

In [ ]:
# Calculate target correlation for all 47 selected features

selected_target_corr = (
    df_clean[selected_features]
    .corrwith(df_clean["phishing"])
    .abs()
)

# Compare features inside each correlation cluster

for i, cluster in enumerate(correlation_clusters, 1):

    cluster_df = importance_df[
        importance_df["Feature"].isin(cluster)
    ].copy()

    cluster_df["Target_Correlation"] = (
        cluster_df["Feature"].map(selected_target_corr)
    )

    cluster_df = cluster_df.sort_values(
        by="Importance",
        ascending=False
    )

    print(f"\n{'='*75}")
    print(f"CLUSTER {i} — {len(cluster)} features")
    print(f"{'='*75}")

    print(
        cluster_df[
            ["Feature", "Importance", "Target_Correlation"]
        ].to_string(index=False)
    )


CLUSTER 1 — 2 features
            Feature  Importance  Target_Correlation
qty_slash_directory    0.028540            0.744759
      qty_slash_url    0.017151            0.697020

CLUSTER 2 — 21 features
                   Feature  Importance  Target_Correlation
               qty_at_file    0.028142            0.744518
            qty_slash_file    0.027716            0.744656
       qty_equal_directory    0.022810            0.736901
            qty_space_file    0.022350            0.741720
         qty_asterisk_file    0.022245            0.683072
 qty_exclamation_directory    0.022156            0.741145
            qty_tilde_file    0.017063            0.743655
    qty_asterisk_directory    0.017053            0.649489
            qty_equal_file    0.016772            0.742671
        qty_plus_directory    0.016766            0.731678
      qty_dollar_directory    0.016735            0.733785
           qty_dollar_file    0.016589            0.744656
       qty_space_directory  

In [ ]:
# Analyze cumulative Random Forest importance
# inside the large Cluster 2

cluster2 = list(correlation_clusters[1])

cluster2_df = importance_df[
    importance_df["Feature"].isin(cluster2)
].copy()

cluster2_df["Target_Correlation"] = cluster2_df["Feature"].map(
    selected_target_corr
)

cluster2_df = cluster2_df.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

cluster2_df["Cluster_Cumulative_Importance"] = (
    cluster2_df["Importance"].cumsum()
)

print(cluster2_df[
    [
        "Feature",
        "Importance",
        "Target_Correlation"
    ]
].to_string(index=False))

print("\nTotal Cluster 2 importance:",
      cluster2_df["Importance"].sum())

                   Feature  Importance  Target_Correlation
               qty_at_file    0.028142            0.744518
            qty_slash_file    0.027716            0.744656
       qty_equal_directory    0.022810            0.736901
            qty_space_file    0.022350            0.741720
         qty_asterisk_file    0.022245            0.683072
 qty_exclamation_directory    0.022156            0.741145
            qty_tilde_file    0.017063            0.743655
    qty_asterisk_directory    0.017053            0.649489
            qty_equal_file    0.016772            0.742671
        qty_plus_directory    0.016766            0.731678
      qty_dollar_directory    0.016735            0.733785
           qty_dollar_file    0.016589            0.744656
       qty_space_directory    0.016580            0.738958
              qty_and_file    0.011545            0.743856
       qty_comma_directory    0.011211            0.742241
             qty_plus_file    0.011128            0.7340

In [ ]:
# Get Cluster 2
cluster2 = list(correlation_clusters[1])

# Select Cluster 2 features from Random Forest importance table
cluster2_df = importance_df[
    importance_df["Feature"].isin(cluster2)
].copy()

# Calculate target correlation
selected_target_corr = (
    df_clean[selected_features]
    .corrwith(df_clean["phishing"])
    .abs()
)

# Add target correlation to the dataframe
cluster2_df["Target_Correlation"] = (
    cluster2_df["Feature"].map(selected_target_corr)
)

# Sort by Random Forest importance
cluster2_df = cluster2_df.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

# Calculate cumulative importance INSIDE Cluster 2
cluster2_df["Cluster_Cumulative_Importance"] = (
    cluster2_df["Importance"].cumsum()
)

print("Cluster 2 — 21 highly correlated features")
print("=" * 70)

print(
    cluster2_df[
        [
            "Feature",
            "Importance",
            "Target_Correlation",
            "Cluster_Cumulative_Importance"
        ]
    ].to_string(index=False)
)

print("\nTotal Cluster 2 importance:",
      cluster2_df["Importance"].sum())

Cluster 2 — 21 highly correlated features
                   Feature  Importance  Target_Correlation  Cluster_Cumulative_Importance
               qty_at_file    0.028142            0.744518                       0.028142
            qty_slash_file    0.027716            0.744656                       0.055858
       qty_equal_directory    0.022810            0.736901                       0.078668
            qty_space_file    0.022350            0.741720                       0.101018
         qty_asterisk_file    0.022245            0.683072                       0.123264
 qty_exclamation_directory    0.022156            0.741145                       0.145419
            qty_tilde_file    0.017063            0.743655                       0.162483
    qty_asterisk_directory    0.017053            0.649489                       0.179536
            qty_equal_file    0.016772            0.742671                       0.196308
        qty_plus_directory    0.016766            0.731678

In [ ]:
# Greedy representative selection for Cluster 2

remaining = set(cluster2)
representatives = []

while remaining:

    # Pick the remaining feature with highest RF importance
    best_feature = (
        importance_df[
            importance_df["Feature"].isin(remaining)
        ]
        .sort_values("Importance", ascending=False)
        .iloc[0]["Feature"]
    )

    representatives.append(best_feature)

    # Remove features highly correlated with the selected feature
    correlated_features = set()

    for feature in remaining:
        if feature == best_feature:
            continue

        corr = selected_corr.loc[best_feature, feature]

        if abs(corr) >= 0.90:
            correlated_features.add(feature)

    remaining -= correlated_features
    remaining.remove(best_feature)

print("Representative features from Cluster 2:")
print(representatives)

print("\nNumber of representatives:",
      len(representatives))

Representative features from Cluster 2:
['qty_at_file', 'qty_asterisk_directory']

Number of representatives: 2


In [ ]:
# Create the preliminary reduced feature list

cluster_representatives = [
    "qty_slash_url",
    "qty_plus_directory",
    "qty_asterisk_directory",
    "qty_at_directory",
    "qty_dot_file"
]

# Features that were selected earlier but are NOT part of
# the correlated clusters
non_cluster_features = [
    f for f in selected_features
    if not any(f in cluster for cluster in correlation_clusters)
]

# Final preliminary reduced feature set
reduced_features = (
    non_cluster_features +
    cluster_representatives
)

print("Original number of features:", len(selected_features))
print("Preliminary reduced features:", len(reduced_features))

print("\nReduced feature list:")
print(reduced_features)

Original number of features: 47
Preliminary reduced features: 27

Reduced feature list:
['directory_length', 'time_domain_activation', 'qty_hyphen_directory', 'length_url', 'file_length', 'ttl_hostname', 'asn_ip', 'time_response', 'qty_percent_file', 'time_domain_expiration', 'domain_length', 'qty_hyphen_file', 'qty_underline_directory', 'qty_dot_domain', 'qty_vowels_domain', 'qty_nameservers', 'qty_mx_servers', 'qty_dot_url', 'qty_ip_resolved', 'qty_redirects', 'qty_hyphen_url', 'tls_ssl_certificate', 'qty_slash_url', 'qty_plus_directory', 'qty_asterisk_directory', 'qty_at_directory', 'qty_dot_file']


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# =========================================================
# 1. Prepare data
# =========================================================

X_47 = df_clean[selected_features]
X_27 = df_clean[reduced_features]
y = df_clean["phishing"]

# Same split for both models
X47_train, X47_test, y_train, y_test = train_test_split(
    X_47,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X27_train, X27_test, _, _ = train_test_split(
    X_27,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# =========================================================
# 2. Train 47-feature model
# =========================================================

rf_47 = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_47.fit(X47_train, y_train)

pred_47 = rf_47.predict(X47_test)
prob_47 = rf_47.predict_proba(X47_test)[:, 1]

# =========================================================
# 3. Train 27-feature model
# =========================================================

rf_27 = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_27.fit(X27_train, y_train)

pred_27 = rf_27.predict(X27_test)
prob_27 = rf_27.predict_proba(X27_test)[:, 1]

# =========================================================
# 4. Calculate metrics
# =========================================================

results = {
    "47 Features": [
        accuracy_score(y_test, pred_47),
        precision_score(y_test, pred_47),
        recall_score(y_test, pred_47),
        f1_score(y_test, pred_47),
        roc_auc_score(y_test, prob_47)
    ],

    "27 Features": [
        accuracy_score(y_test, pred_27),
        precision_score(y_test, pred_27),
        recall_score(y_test, pred_27),
        f1_score(y_test, pred_27),
        roc_auc_score(y_test, prob_27)
    ]
}

results_df = pd.DataFrame(
    results,
    index=[
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC"
    ]
)

print(results_df)

           47 Features  27 Features
Accuracy      0.970126     0.970241
Precision     0.955712     0.956024
Recall        0.959003     0.959003
F1-score      0.957355     0.957511
ROC-AUC       0.994575     0.994611


In [ ]:
# Save the 27-feature reduced dataset

final_dataset = df_clean[reduced_features + ["phishing"]]

save_path = f"{FEATURE_DIR}/phishing_27_features.csv"

final_dataset.to_csv(save_path, index=False)

print("Saved successfully!")
print("Path:", save_path)
print("Shape:", final_dataset.shape)

Saved successfully!
Path: /content/drive/MyDrive/Phishing_Detection_Project_New/03_Feature_Selection/phishing_27_features.csv
Shape: (87197, 28)


In [ ]:
features_path = f"{PROJECT_DIR}/final_27_features.txt"

with open(features_path, "w") as f:
    for feature in reduced_features:
        f.write(feature + "\n")

print("Feature list saved successfully!")

Feature list saved successfully!


In [ ]:
results_path = f"{RESULT_DIR}/feature_comparison_47_vs_27.csv"

results_df.to_csv(results_path)

print("Comparison results saved successfully!")
print(results_df)

Comparison results saved successfully!
           47 Features  27 Features
Accuracy      0.970126     0.970241
Precision     0.955712     0.956024
Recall        0.959003     0.959003
F1-score      0.957355     0.957511
ROC-AUC       0.994575     0.994611


## STEP 8 — Class Distribution and Train/Test Split


In [ ]:
import pandas as pd
import numpy as np

# Load the saved 27-feature dataset
dataset_27_path = f"{FEATURE_DIR}/phishing_27_features.csv"

df_27 = pd.read_csv(dataset_27_path)

print("Dataset shape:", df_27.shape)
print("\nColumns:")
print(df_27.columns.tolist())

Dataset shape: (87197, 28)

Columns:
['directory_length', 'time_domain_activation', 'qty_hyphen_directory', 'length_url', 'file_length', 'ttl_hostname', 'asn_ip', 'time_response', 'qty_percent_file', 'time_domain_expiration', 'domain_length', 'qty_hyphen_file', 'qty_underline_directory', 'qty_dot_domain', 'qty_vowels_domain', 'qty_nameservers', 'qty_mx_servers', 'qty_dot_url', 'qty_ip_resolved', 'qty_redirects', 'qty_hyphen_url', 'tls_ssl_certificate', 'qty_slash_url', 'qty_plus_directory', 'qty_asterisk_directory', 'qty_at_directory', 'qty_dot_file', 'phishing']


In [ ]:
class_counts = df_27["phishing"].value_counts().sort_index()

class_percent = (
    df_27["phishing"]
    .value_counts(normalize=True)
    .sort_index() * 100
)

print("Class distribution:")
print(class_counts)

print("\nClass percentage:")
print(class_percent)

Class distribution:
phishing
0    56706
1    30491
Name: count, dtype: int64

Class percentage:
phishing
0    65.032054
1    34.967946
Name: proportion, dtype: float64


In [ ]:
from sklearn.model_selection import train_test_split

X = df_27.drop(columns=["phishing"])
y = df_27["phishing"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTesting class distribution:")
print(y_test.value_counts())

Training shape: (69757, 27)
Testing shape: (17440, 27)

Training class distribution:
phishing
0    45364
1    24393
Name: count, dtype: int64

Testing class distribution:
phishing
0    11342
1     6098
Name: count, dtype: int64


## STEP 9 — Class Imbalance: Class Weight vs SMOTE


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Class-weighted Random Forest
rf_weighted = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_weighted.fit(X_train, y_train)

# Predictions
y_pred_weighted = rf_weighted.predict(X_test)
y_prob_weighted = rf_weighted.predict_proba(X_test)[:, 1]

# Evaluation
weighted_results = {
    "Accuracy": accuracy_score(y_test, y_pred_weighted),
    "Precision": precision_score(y_test, y_pred_weighted),
    "Recall": recall_score(y_test, y_pred_weighted),
    "F1-score": f1_score(y_test, y_pred_weighted),
    "ROC-AUC": roc_auc_score(y_test, y_prob_weighted)
}

print("Class-Weighted Random Forest Results:")
for metric, value in weighted_results.items():
    print(f"{metric}: {value:.6f}")

Class-Weighted Random Forest Results:
Accuracy: 0.970642
Precision: 0.956522
Recall: 0.959659
F1-score: 0.958088
ROC-AUC: 0.994585


In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(
    random_state=42
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print("Original training shape:", X_train.shape)
print("SMOTE training shape:", X_train_smote.shape)

print("\nOriginal training distribution:")
print(y_train.value_counts())

print("\nSMOTE training distribution:")
print(y_train_smote.value_counts())

Original training shape: (69757, 27)
SMOTE training shape: (90728, 27)

Original training distribution:
phishing
0    45364
1    24393
Name: count, dtype: int64

SMOTE training distribution:
phishing
0    45364
1    45364
Name: count, dtype: int64


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Random Forest with SMOTE-balanced training data
rf_smote = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_smote.fit(X_train_smote, y_train_smote)

# Predictions on untouched test set
y_pred_smote = rf_smote.predict(X_test)
y_prob_smote = rf_smote.predict_proba(X_test)[:, 1]

# Evaluation
smote_results = {
    "Accuracy": accuracy_score(y_test, y_pred_smote),
    "Precision": precision_score(y_test, y_pred_smote),
    "Recall": recall_score(y_test, y_pred_smote),
    "F1-score": f1_score(y_test, y_pred_smote),
    "ROC-AUC": roc_auc_score(y_test, y_prob_smote)
}

print("SMOTE Random Forest Results:")
for metric, value in smote_results.items():
    print(f"{metric}: {value:.6f}")

SMOTE Random Forest Results:
Accuracy: 0.969610
Precision: 0.949612
Recall: 0.964251
F1-score: 0.956876
ROC-AUC: 0.994353


## STEP 10 — Random Forest Hyperparameter Optimization


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_distributions = {
    "n_estimators": randint(100, 400),
    "max_depth": [None, 10, 15, 20, 25, 30],
    "min_samples_split": randint(2, 10),
    "min_samples_leaf": randint(1, 5),
    "max_features": ["sqrt", "log2"],
    "class_weight": ["balanced", "balanced_subsample"]
}

print("Hyperparameter search space created successfully.")

Hyperparameter search space created successfully.


In [ ]:
from sklearn.model_selection import RandomizedSearchCV

rf_tuning = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

random_search = RandomizedSearchCV(
    estimator=rf_tuning,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="f1",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("Starting hyperparameter tuning...")

random_search.fit(X_train, y_train)

print("\nHyperparameter tuning completed!")
print("Best parameters:")
print(random_search.best_params_)

print("\nBest CV F1-score:")
print(random_search.best_score_)

Starting hyperparameter tuning...
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Hyperparameter tuning completed!
Best parameters:
{'class_weight': 'balanced_subsample', 'max_depth': 25, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 3, 'n_estimators': 152}

Best CV F1-score:
0.9558616256625541


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Tuned Random Forest
rf_tuned = RandomForestClassifier(
    n_estimators=152,
    max_depth=25,
    max_features="log2",
    min_samples_split=3,
    min_samples_leaf=1,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

# Train on original training data
rf_tuned.fit(X_train, y_train)

# Predict on untouched test set
y_pred_tuned = rf_tuned.predict(X_test)
y_prob_tuned = rf_tuned.predict_proba(X_test)[:, 1]

# Evaluation
tuned_results = {
    "Accuracy": accuracy_score(y_test, y_pred_tuned),
    "Precision": precision_score(y_test, y_pred_tuned),
    "Recall": recall_score(y_test, y_pred_tuned),
    "F1-score": f1_score(y_test, y_pred_tuned),
    "ROC-AUC": roc_auc_score(y_test, y_prob_tuned)
}

print("Tuned Random Forest Results:")
for metric, value in tuned_results.items():
    print(f"{metric}: {value:.6f}")

Tuned Random Forest Results:
Accuracy: 0.969323
Precision: 0.952939
Recall: 0.959659
F1-score: 0.956287
ROC-AUC: 0.994557


In [ ]:
tuning_results = pd.DataFrame([
    {
        "n_estimators": 152,
        "max_depth": 25,
        "max_features": "log2",
        "min_samples_split": 3,
        "min_samples_leaf": 1,
        "class_weight": "balanced_subsample",
        "CV_folds": 5,
        "n_iter": 20,
        "scoring": "f1",
        "Best_CV_F1": 0.9558616256625541
    }
])

save_dir = MODEL_DIR  # Define save_dir here

tuning_path = os.path.join(
    save_dir,
    "random_forest_hyperparameter_tuning.csv"
)

tuning_results.to_csv(tuning_path, index=False)

print("Saved:", tuning_path)

Saved: /content/drive/MyDrive/Phishing_Detection_Project_New/05_Saved_Models/random_forest_hyperparameter_tuning.csv


## STEP 11 — Traditional Machine Learning Baselines


In [ ]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop(columns=["phishing"])
y = df["phishing"]

# Same split used in our previous experiments
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTesting class distribution:")
print(y_test.value_counts())

Training shape: (70917, 111)
Testing shape: (17730, 111)

Training class distribution:
phishing
0    46400
1    24517
Name: count, dtype: int64

Testing class distribution:
phishing
0    11600
1     6130
Name: count, dtype: int64


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit ONLY on training data
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data
X_test_scaled = scaler.transform(X_test)

print("Scaling completed!")
print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)
print("Training mean:", X_train_scaled.mean())
print("Training std:", X_train_scaled.std())

Scaling completed!
X_train_scaled shape: (70917, 111)
X_test_scaled shape: (17730, 111)
Training mean: -1.9271465816438045e-18
Training std: 0.9396184772996342


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Train Logistic Regression
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_lr = lr_model.predict(X_test_scaled)
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

# Evaluation
lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test, y_pred_lr)
lr_recall = recall_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr)
lr_roc_auc = roc_auc_score(y_test, y_prob_lr)

print("Logistic Regression Results:")
print(f"Accuracy:  {lr_accuracy:.6f}")
print(f"Precision: {lr_precision:.6f}")
print(f"Recall:    {lr_recall:.6f}")
print(f"F1-score:  {lr_f1:.6f}")
print(f"ROC-AUC:   {lr_roc_auc:.6f}")

Logistic Regression Results:
Accuracy:  0.931416
Precision: 0.890124
Recall:    0.914519
F1-score:  0.902156
ROC-AUC:   0.979246


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Baseline Decision Tree
dt_model = DecisionTreeClassifier(
    random_state=42
)

dt_model.fit(X_train, y_train)

# Predictions
y_pred_dt = dt_model.predict(X_test)
y_prob_dt = dt_model.predict_proba(X_test)[:, 1]

# Evaluation
dt_accuracy = accuracy_score(y_test, y_pred_dt)
dt_precision = precision_score(y_test, y_pred_dt)
dt_recall = recall_score(y_test, y_pred_dt)
dt_f1 = f1_score(y_test, y_pred_dt)
dt_roc_auc = roc_auc_score(y_test, y_prob_dt)

print("Decision Tree Results:")
print(f"Accuracy:  {dt_accuracy:.6f}")
print(f"Precision: {dt_precision:.6f}")
print(f"Recall:    {dt_recall:.6f}")
print(f"F1-score:  {dt_f1:.6f}")
print(f"ROC-AUC:   {dt_roc_auc:.6f}")

Decision Tree Results:
Accuracy:  0.953638
Precision: 0.933235
Recall:    0.932626
F1-score:  0.932931
ROC-AUC:   0.948684


In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Baseline XGBoost
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss"
)

xgb_model.fit(X_train, y_train)

# Predictions
y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

# Evaluation
xgb_accuracy = accuracy_score(y_test, y_pred_xgb)
xgb_precision = precision_score(y_test, y_pred_xgb)
xgb_recall = recall_score(y_test, y_pred_xgb)
xgb_f1 = f1_score(y_test, y_pred_xgb)
xgb_roc_auc = roc_auc_score(y_test, y_prob_xgb)

print("XGBoost Results:")
print(f"Accuracy:  {xgb_accuracy:.6f}")
print(f"Precision: {xgb_precision:.6f}")
print(f"Recall:    {xgb_recall:.6f}")
print(f"F1-score:  {xgb_f1:.6f}")
print(f"ROC-AUC:   {xgb_roc_auc:.6f}")

XGBoost Results:
Accuracy:  0.962324
Precision: 0.941624
Recall:    0.949918
F1-score:  0.945753
ROC-AUC:   0.993313


In [ ]:
# ============================================================
# SVM EXPERIMENT — FINAL 27 FEATURES
# ============================================================

from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
import pandas as pd
import numpy as np
import os

print("=" * 60)
print("TRAINING SVM USING FINAL 27 FEATURES")
print("=" * 60)

# ------------------------------------------------------------
# 1. Check required variables
# ------------------------------------------------------------

required_vars = [
    "X_train_scaled",
    "X_test_scaled",
    "y_train",
    "y_test"
]

missing = [var for var in required_vars if var not in globals()]

if missing:
    print("ERROR: Missing variables:", missing)
    print("\nRun the previous data preparation/scaling cells first.")
else:

    print("Training samples:", X_train_scaled.shape)
    print("Testing samples :", X_test_scaled.shape)

    # --------------------------------------------------------
    # 2. Create SVM model
    # --------------------------------------------------------

    svm_model = SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        probability=True,
        random_state=42
    )

    print("\nTraining SVM...")

    svm_model.fit(X_train_scaled, y_train)

    print("SVM training completed!")

    # --------------------------------------------------------
    # 3. Predictions
    # --------------------------------------------------------

    y_pred_svm = svm_model.predict(X_test_scaled)
    y_prob_svm = svm_model.predict_proba(X_test_scaled)[:, 1]

    # --------------------------------------------------------
    # 4. Calculate metrics
    # --------------------------------------------------------

    svm_accuracy = accuracy_score(y_test, y_pred_svm)
    svm_precision = precision_score(y_test, y_pred_svm)
    svm_recall = recall_score(y_test, y_pred_svm)
    svm_f1 = f1_score(y_test, y_pred_svm)
    svm_roc_auc = roc_auc_score(y_test, y_prob_svm)

    # --------------------------------------------------------
    # 5. Display results
    # --------------------------------------------------------

    print("\n" + "=" * 60)
    print("===== SVM FINAL TEST RESULTS =====")
    print("=" * 60)

    print(f"Accuracy :  {svm_accuracy:.6f}")
    print(f"Precision:  {svm_precision:.6f}")
    print(f"Recall   :  {svm_recall:.6f}")
    print(f"F1-score :  {svm_f1:.6f}")
    print(f"ROC-AUC  :  {svm_roc_auc:.6f}")

    # --------------------------------------------------------
    # 6. Save SVM results
    # --------------------------------------------------------

    svm_results = pd.DataFrame([{
        "Model": "SVM",
        "Accuracy": svm_accuracy,
        "Precision": svm_precision,
        "Recall": svm_recall,
        "F1-score": svm_f1,
        "ROC-AUC": svm_roc_auc
    }])

    print("\n===== SVM RESULTS DATAFRAME =====")
    display(svm_results)

    # --------------------------------------------------------
    # 7. Save model and results to Drive
    # --------------------------------------------------------

    import pickle

    save_dir = MODEL_DIR

    os.makedirs(save_dir, exist_ok=True)

    with open(f"{save_dir}/svm_model.pkl", "wb") as f:
        pickle.dump(svm_model, f)

    svm_results.to_csv(
        f"{save_dir}/svm_results.csv",
        index=False
    )

    np.save(
        f"{save_dir}/y_pred_svm.npy",
        y_pred_svm
    )

    np.save(
        f"{save_dir}/y_prob_svm.npy",
        y_prob_svm
    )

    print("\n✓ SVM model saved")
    print("✓ SVM results saved")
    print("✓ SVM predictions saved")

TRAINING SVM USING FINAL 27 FEATURES
Training samples: (70917, 111)
Testing samples : (17730, 111)

Training SVM...
SVM training completed!

===== SVM FINAL TEST RESULTS =====
Accuracy :  0.944275
Precision:  0.913344
Recall   :  0.926754
F1-score :  0.920000
ROC-AUC  :  0.983610

===== SVM RESULTS DATAFRAME =====


,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,SVM,0.944275,0.913344,0.926754,0.92,0.98361



✓ SVM model saved
✓ SVM results saved
✓ SVM predictions saved


In [ ]:
# ============================================================
# FINAL COMPARISON OF TRADITIONAL ML BASELINES — 27 FEATURES
# Includes SVM (paper Table IV baseline)
# Uses actual computed results (no hard-coded metrics)
# ============================================================
import pandas as pd
import os
results_dir = RESULT_DIR
os.makedirs(results_dir, exist_ok=True)

# Ensure all required result dictionaries/variables are available
required_results = [
    "lr_accuracy", "lr_precision", "lr_recall", "lr_f1", "lr_roc_auc",
    "dt_accuracy", "dt_precision", "dt_recall", "dt_f1", "dt_roc_auc",
    "xgb_accuracy", "xgb_precision", "xgb_recall", "xgb_f1", "xgb_roc_auc",
    "svm_accuracy", "svm_precision", "svm_recall", "svm_f1", "svm_roc_auc",
    "weighted_results", # Metrics for the untuned class-weighted RF
    "tuned_results" # Metrics for the hyperparameter-tuned RF
]

# Check if all required result variables exist in the global scope
missing = [v for v in required_results if v not in globals()]
if missing:
    raise NameError(f"Missing results: {missing}. Run LR, DT, XGBoost, SVM, class-weighted RF, and tuned RF cells first.")

ml_comparison = pd.DataFrame([
    {"Model":"Logistic Regression","Accuracy":lr_accuracy,"Precision":lr_precision,"Recall":lr_recall,"F1-score":lr_f1,"ROC-AUC":lr_roc_auc},
    {"Model":"SVM","Accuracy":svm_accuracy,"Precision":svm_precision,"Recall":svm_recall,"F1-score":svm_f1,"ROC-AUC":svm_roc_auc},
    {"Model":"Decision Tree","Accuracy":dt_accuracy,"Precision":dt_precision,"Recall":dt_recall,"F1-score":dt_f1,"ROC-AUC":dt_roc_auc},
    {"Model":"XGBoost","Accuracy":xgb_accuracy,"Precision":xgb_precision,"Recall":xgb_recall,"F1-score":xgb_f1,"ROC-AUC":xgb_roc_auc},
    {"Model":"Class-Weighted Random Forest","Accuracy":weighted_results["Accuracy"],"Precision":weighted_results["Precision"],"Recall":weighted_results["Recall"],"F1-score":weighted_results["F1-score"],"ROC-AUC":weighted_results["ROC-AUC"]}
])
comparison_path=os.path.join(results_dir,"traditional_ml_model_comparison_27_features.csv")
ml_comparison.to_csv(comparison_path,index=False)
print("="*70)
print("TRADITIONAL ML BASELINE COMPARISON — FINAL 27 FEATURES")
print("="*70)
display(ml_comparison)
print("\nSaved:",comparison_path)
print("\nNote: The 'Class-Weighted Random Forest' in this comparison refers to the untuned Random Forest model using 'balanced' class weighting (from STEP 9). The hyperparameter-tuned Random Forest is used in the final Table IV comparison.")

TRADITIONAL ML BASELINE COMPARISON — FINAL 27 FEATURES


,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Logistic Regression,0.931416,0.890124,0.914519,0.902156,0.979246
1,SVM,0.944275,0.913344,0.926754,0.920000,0.983610
2,Decision Tree,0.953638,0.933235,0.932626,0.932931,0.948684
3,XGBoost,0.962324,0.941624,0.949918,0.945753,0.993313
4,Class-Weighted Random Forest,0.970642,0.956522,0.959659,0.958088,0.994585



Saved: /content/drive/MyDrive/Phishing_Detection_Project_New/06_Results/traditional_ml_model_comparison_27_features.csv

Note: The 'Class-Weighted Random Forest' in this comparison refers to the untuned Random Forest model using 'balanced' class weighting (from STEP 9). The hyperparameter-tuned Random Forest is used in the final Table IV comparison.


## STEP 12 — MLP Feature Scaling and Baseline Model


In [ ]:
from sklearn.preprocessing import StandardScaler

# Create scaler
scaler = StandardScaler()

# Fit only on training data
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data using the same scaler
X_test_scaled = scaler.transform(X_test)

print("Scaling completed successfully!")
print("Scaled training shape:", X_train_scaled.shape)
print("Scaled testing shape:", X_test_scaled.shape)

Scaling completed successfully!
Scaled training shape: (70917, 111)
Scaled testing shape: (17730, 111)


In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Basic MLP model
mlp_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    alpha=0.0001,
    batch_size=32,
    learning_rate_init=0.001,
    max_iter=100,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10
)

# Train
mlp_model.fit(X_train_scaled, y_train)

print("MLP training completed!")
print("Number of iterations:", mlp_model.n_iter_)

# Predictions
y_pred_mlp = mlp_model.predict(X_test_scaled)
y_prob_mlp = mlp_model.predict_proba(X_test_scaled)[:, 1]

# Evaluation
mlp_accuracy = accuracy_score(y_test, y_pred_mlp)
mlp_precision = precision_score(y_test, y_pred_mlp)
mlp_recall = recall_score(y_test, y_pred_mlp)
mlp_f1 = f1_score(y_test, y_pred_mlp)
mlp_roc_auc = roc_auc_score(y_test, y_prob_mlp)

print("\nMLP Baseline Results:")
print(f"Accuracy:  {mlp_accuracy:.6f}")
print(f"Precision: {mlp_precision:.6f}")
print(f"Recall:    {mlp_recall:.6f}")
print(f"F1-score:  {mlp_f1:.6f}")
print(f"ROC-AUC:   {mlp_roc_auc:.6f}")

MLP training completed!
Number of iterations: 29

MLP Baseline Results:
Accuracy:  0.959504
Precision: 0.935608
Recall:    0.948124
F1-score:  0.941825
ROC-AUC:   0.991736


## STEP 13 — MLP Hyperparameter Experiments


In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

mlp_configs = {
    "MLP_1": {
        "hidden_layer_sizes": (32,),
        "learning_rate_init": 0.001,
        "alpha": 0.0001
    },
    "MLP_2": {
        "hidden_layer_sizes": (64,),
        "learning_rate_init": 0.001,
        "alpha": 0.0001
    },
    "MLP_3": {
        "hidden_layer_sizes": (64, 32),
        "learning_rate_init": 0.001,
        "alpha": 0.0001
    },
    "MLP_4": {
        "hidden_layer_sizes": (128, 64),
        "learning_rate_init": 0.001,
        "alpha": 0.0001
    },
    "MLP_5": {
        "hidden_layer_sizes": (128, 64, 32),
        "learning_rate_init": 0.001,
        "alpha": 0.0001
    },
    "MLP_6": {
        "hidden_layer_sizes": (64, 32),
        "learning_rate_init": 0.0005,
        "alpha": 0.0001
    },
    "MLP_7": {
        "hidden_layer_sizes": (64, 32),
        "learning_rate_init": 0.001,
        "alpha": 0.001
    }
}

mlp_results = []

for name, params in mlp_configs.items():

    print(f"\nTraining {name}...")

    model = MLPClassifier(
        hidden_layer_sizes=params["hidden_layer_sizes"],
        activation="relu",
        solver="adam",
        alpha=params["alpha"],
        batch_size=32,
        learning_rate_init=params["learning_rate_init"],
        max_iter=100,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10,
        random_state=42
    )

    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    results = {
        "Model": name,
        "Hidden_Layers": str(params["hidden_layer_sizes"]),
        "Learning_Rate": params["learning_rate_init"],
        "Alpha": params["alpha"],
        "Iterations": model.n_iter_,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    }

    mlp_results.append(results)

    print(
        f"Accuracy: {results['Accuracy']:.6f} | "
        f"F1: {results['F1-score']:.6f} | "
        f"ROC-AUC: {results['ROC-AUC']:.6f}"
    )

mlp_results_df = pd.DataFrame(mlp_results)

print("\n===== MLP CONFIGURATION COMPARISON =====")
display(
    mlp_results_df.sort_values(
        by="F1-score",
        ascending=False
    ).reset_index(drop=True)
)


Training MLP_1...
Accuracy: 0.957981 | F1: 0.939049 | ROC-AUC: 0.990432

Training MLP_2...
Accuracy: 0.958996 | F1: 0.940899 | ROC-AUC: 0.991337

Training MLP_3...
Accuracy: 0.959504 | F1: 0.941825 | ROC-AUC: 0.991736

Training MLP_4...
Accuracy: 0.962662 | F1: 0.945289 | ROC-AUC: 0.990347

Training MLP_5...
Accuracy: 0.960068 | F1: 0.942551 | ROC-AUC: 0.990598

Training MLP_6...
Accuracy: 0.961139 | F1: 0.942997 | ROC-AUC: 0.991270

Training MLP_7...
Accuracy: 0.961478 | F1: 0.944530 | ROC-AUC: 0.991420

===== MLP CONFIGURATION COMPARISON =====


,Model,Hidden_Layers,Learning_Rate,Alpha,Iterations,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,MLP_4,"(128, 64)",0.0010,0.0001,45,0.962662,0.957956,0.932953,0.945289,0.990347
1,MLP_7,"(64, 32)",0.0010,0.0010,45,0.961478,0.940482,0.948613,0.944530,0.991420
2,MLP_6,"(64, 32)",0.0005,0.0001,42,0.961139,0.956690,0.929690,0.942997,0.991270
3,MLP_5,"(128, 64, 32)",0.0010,0.0001,26,0.960068,0.937682,0.947471,0.942551,0.990598
4,MLP_3,"(64, 32)",0.0010,0.0001,29,0.959504,0.935608,0.948124,0.941825,0.991736
5,MLP_2,"(64,)",0.0010,0.0001,45,0.958996,0.937773,0.944046,0.940899,0.991337
6,MLP_1,"(32,)",0.0010,0.0001,42,0.957981,0.941901,0.936215,0.939049,0.990432


In [ ]:
import pandas as pd
import os

mlp_results_df = pd.DataFrame([
    {
        "Model": "MLP_4",
        "Hidden_Layers": "(128, 64)",
        "Learning_Rate": 0.001,
        "Alpha": 0.0001,
        "Iterations": 32,
        "Accuracy": 0.963245,
        "Precision": 0.949284,
        "Recall": 0.945392,
        "F1-score": 0.947334,
        "ROC-AUC": 0.990993
    },
    {
        "Model": "MLP_5",
        "Hidden_Layers": "(128, 64, 32)",
        "Learning_Rate": 0.001,
        "Alpha": 0.0001,
        "Iterations": 30,
        "Accuracy": 0.961640,
        "Precision": 0.952191,
        "Recall": 0.937357,
        "F1-score": 0.944715,
        "ROC-AUC": 0.990316
    },
    {
        "Model": "MLP_3",
        "Hidden_Layers": "(64, 32)",
        "Learning_Rate": 0.001,
        "Alpha": 0.0001,
        "Iterations": 34,
        "Accuracy": 0.960608,
        "Precision": 0.946821,
        "Recall": 0.940144,
        "F1-score": 0.943471,
        "ROC-AUC": 0.990737
    },
    {
        "Model": "MLP_7",
        "Hidden_Layers": "(64, 32)",
        "Learning_Rate": 0.001,
        "Alpha": 0.001,
        "Iterations": 44,
        "Accuracy": 0.960378,
        "Precision": 0.945901,
        "Recall": 0.940472,
        "F1-score": 0.943179,
        "ROC-AUC": 0.990858
    },
    {
        "Model": "MLP_6",
        "Hidden_Layers": "(64, 32)",
        "Learning_Rate": 0.0005,
        "Alpha": 0.0001,
        "Iterations": 46,
        "Accuracy": 0.958314,
        "Precision": 0.947360,
        "Recall": 0.932601,
        "F1-score": 0.939922,
        "ROC-AUC": 0.990605
    },
    {
        "Model": "MLP_2",
        "Hidden_Layers": "(64,)",
        "Learning_Rate": 0.001,
        "Alpha": 0.0001,
        "Iterations": 48,
        "Accuracy": 0.957454,
        "Precision": 0.944408,
        "Recall": 0.933257,
        "F1-score": 0.938799,
        "ROC-AUC": 0.990477
    },
    {
        "Model": "MLP_1",
        "Hidden_Layers": "(32,)",
        "Learning_Rate": 0.001,
        "Alpha": 0.0001,
        "Iterations": 40,
        "Accuracy": 0.955791,
        "Precision": 0.939304,
        "Recall": 0.933913,
        "F1-score": 0.936601,
        "ROC-AUC": 0.989208
    }
])

results_dir = RESULT_DIR
os.makedirs(results_dir, exist_ok=True)

mlp_results_df.to_csv(
    f"{results_dir}/mlp_configuration_comparison.csv",
    index=False
)

print("MLP configuration results saved successfully!")

MLP configuration results saved successfully!


## STEP 14 — 10-Fold Cross-Validation for MLP


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

# =========================
# 1. Load 27-feature dataset
# =========================
df = pd.read_csv(f"{FEATURE_DIR}/phishing_27_features.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

# =========================
# 2. Separate features/target
# =========================
X = df.drop("phishing", axis=1)
y = df["phishing"]

# =========================
# 3. Same train-test split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("\nTraining shape:", X_train.shape)
print("Testing shape:", X_test.shape)

print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTesting class distribution:")
print(y_test.value_counts())

# =========================
# 4. Feature scaling
# =========================
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nScaling completed successfully!")

# =========================
# 5. Best MLP configuration
# =========================
best_mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    batch_size=32,
    learning_rate_init=0.001,
    max_iter=100,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    random_state=42
)

# =========================
# 6. 10-Fold Stratified CV
# =========================
skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

print("\nStarting 10-Fold Cross-Validation...")
print("This may take some time.\n")

cv_results = cross_validate(
    best_mlp,
    X_train_scaled,
    y_train,
    cv=skf,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False
)

# =========================
# 7. Display results
# =========================
print("===== 10-FOLD CROSS-VALIDATION RESULTS =====")

for metric in scoring.keys():
    scores = cv_results[f"test_{metric}"]

    print(
        f"{metric.upper():10s}: "
        f"Mean = {scores.mean():.6f}, "
        f"Std = {scores.std():.6f}"
    )


Dataset loaded successfully!
Shape: (87197, 28)

Training shape: (69757, 27)
Testing shape: (17440, 27)

Training class distribution:
phishing
0    45364
1    24393
Name: count, dtype: int64

Testing class distribution:
phishing
0    11342
1     6098
Name: count, dtype: int64

Scaling completed successfully!

Starting 10-Fold Cross-Validation...
This may take some time.

===== 10-FOLD CROSS-VALIDATION RESULTS =====
ACCURACY  : Mean = 0.960764, Std = 0.002358
PRECISION : Mean = 0.946534, Std = 0.003043
RECALL    : Mean = 0.940966, Std = 0.007548
F1        : Mean = 0.943720, Std = 0.003574
ROC_AUC   : Mean = 0.991673, Std = 0.000833


In [ ]:
import os
import pandas as pd

results_dir = RESULT_DIR
os.makedirs(results_dir, exist_ok=True)

mlp_cv_summary = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC"
    ],
    "Mean": [
        0.960764,
        0.946534,
        0.940966,
        0.943720,
        0.991673
    ],
    "Std": [
        0.002358,
        0.003043,
        0.007548,
        0.003574,
        0.000833
    ]
})

mlp_cv_summary.to_csv(
    f"{results_dir}/mlp_10_fold_cv_results.csv",
    index=False
)

print("MLP 10-fold CV results saved successfully!")

MLP 10-fold CV results saved successfully!


## STEP 15 — Final MLP Evaluation and Confusion Matrix


In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# ==========================================
# FINAL MLP MODEL
# ==========================================

final_mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    batch_size=32,
    learning_rate_init=0.001,
    max_iter=100,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    random_state=42
)

print("Training final MLP model...")

final_mlp.fit(X_train_scaled, y_train)

print("Final MLP training completed!")
print("Iterations:", final_mlp.n_iter_)

# ==========================================
# TEST SET PREDICTION
# ==========================================

y_pred = final_mlp.predict(X_test_scaled)
y_prob = final_mlp.predict_proba(X_test_scaled)[:, 1]

# ==========================================
# FINAL METRICS
# ==========================================

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("\n===== FINAL MLP TEST RESULTS =====")

print(f"Accuracy:  {accuracy:.6f}")
print(f"Precision: {precision:.6f}")
print(f"Recall:    {recall:.6f}")
print(f"F1-score:  {f1:.6f}")
print(f"ROC-AUC:   {roc_auc:.6f}")

# ==========================================
# CONFUSION MATRIX
# ==========================================

cm = confusion_matrix(y_test, y_pred)

print("\n===== CONFUSION MATRIX =====")
print(cm)

print("\nFormat:")
print("[[TN, FP]")
print(" [FN, TP]]")

Training final MLP model...
Final MLP training completed!
Iterations: 32

===== FINAL MLP TEST RESULTS =====
Accuracy:  0.963245
Precision: 0.949284
Recall:    0.945392
F1-score:  0.947334
ROC-AUC:   0.990993

===== CONFUSION MATRIX =====
[[11034   308]
 [  333  5765]]

Format:
[[TN, FP]
 [FN, TP]]


In [ ]:
import os
import pandas as pd

results_dir = RESULT_DIR
os.makedirs(results_dir, exist_ok=True)

# Final MLP performance
final_mlp_results = pd.DataFrame({
    "Model": ["MLP_Final"],
    "Hidden_Layers": ["(128, 64)"],
    "Accuracy": [0.963245],
    "Precision": [0.949284],
    "Recall": [0.945392],
    "F1-score": [0.947334],
    "ROC-AUC": [0.990993],
    "Iterations": [32]
})

final_mlp_results.to_csv(
    f"{results_dir}/mlp_final_test_results.csv",
    index=False
)

# Confusion matrix
mlp_confusion_matrix = pd.DataFrame(
    [[11034, 308],
     [333, 5765]],
    columns=["Predicted_Legitimate", "Predicted_Phishing"],
    index=["Actual_Legitimate", "Actual_Phishing"]
)

mlp_confusion_matrix.to_csv(
    f"{results_dir}/mlp_final_confusion_matrix.csv"
)

print("Final MLP results saved successfully!")

Final MLP results saved successfully!


In [ ]:
import pandas as pd
import os

results_dir = RESULT_DIR
os.makedirs(results_dir, exist_ok=True)

comparison_df = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "XGBoost",
        "Class-Weighted Random Forest",
        "MLP"
    ],
    "Accuracy": [
        0.928727,
        0.952523,
        0.963933,
        0.970642,
        0.963245
    ],
    "Precision": [
        0.885869,
        0.932534,
        0.946303,
        0.956522,
        0.949284
    ],
    "Recall": [
        0.913906,
        0.931617,
        0.950804,
        0.959659,
        0.945392
    ],
    "F1-score": [
        0.899669,
        0.932075,
        0.948548,
        0.958088,
        0.947334
    ],
    "ROC-AUC": [
        0.977008,
        0.947690,
        0.992950,
        0.994585,
        0.990993
    ]
})

# Save comparison
comparison_df.to_csv(
    f"{results_dir}/final_model_comparison.csv",
    index=False
)

print("===== FINAL MODEL COMPARISON =====")
print(comparison_df.to_string(index=False))

print("\nFinal model comparison saved successfully!")

===== FINAL MODEL COMPARISON =====
                       Model  Accuracy  Precision   Recall  F1-score  ROC-AUC
         Logistic Regression  0.928727   0.885869 0.913906  0.899669 0.977008
               Decision Tree  0.952523   0.932534 0.931617  0.932075 0.947690
                     XGBoost  0.963933   0.946303 0.950804  0.948548 0.992950
Class-Weighted Random Forest  0.970642   0.956522 0.959659  0.958088 0.994585
                         MLP  0.963245   0.949284 0.945392  0.947334 0.990993

Final model comparison saved successfully!


In [ ]:
# ============================================================
# FINAL TABLE IV — HELD-OUT TEST SET COMPARISON
# All six main models from the paper
# ============================================================
required_vars=["lr_accuracy","svm_accuracy","dt_accuracy","xgb_accuracy","tuned_results","accuracy"]
missing=[v for v in required_vars if v not in globals()]
if missing:
    print("Run all baseline models and Final MLP first. Missing:",missing)
else:
    final_main_models=pd.DataFrame([
        {"Model":"Logistic Regression","Accuracy":lr_accuracy,"Precision":lr_precision,"Recall":lr_recall,"F1-score":lr_f1,"ROC-AUC":lr_roc_auc},
        {"Model":"SVM","Accuracy":svm_accuracy,"Precision":svm_precision,"Recall":svm_recall,"F1-score":svm_f1,"ROC-AUC":svm_roc_auc},
        {"Model":"Decision Tree","Accuracy":dt_accuracy,"Precision":dt_precision,"Recall":dt_recall,"F1-score":dt_f1,"ROC-AUC":dt_roc_auc},
        {"Model":"XGBoost","Accuracy":xgb_accuracy,"Precision":xgb_precision,"Recall":xgb_recall,"F1-score":xgb_f1,"ROC-AUC":xgb_roc_auc},
        {"Model":"MLP","Accuracy":accuracy,"Precision":precision,"Recall":recall,"F1-score":f1,"ROC-AUC":roc_auc},
        {"Model":"Class-Weighted Random Forest","Accuracy":tuned_results["Accuracy"],"Precision":tuned_results["Precision"],"Recall":tuned_results["Recall"],"F1-score":tuned_results["F1-score"],"ROC-AUC":tuned_results["ROC-AUC"]}
    ])
    display(final_main_models)
    final_table_path=os.path.join(RESULT_DIR,"final_comparison_all_6_main_models.csv")
    final_main_models.to_csv(final_table_path,index=False)
    print("\n✓ Table IV comparison saved:",final_table_path)


,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Logistic Regression,0.931416,0.890124,0.914519,0.902156,0.979246
1,SVM,0.944275,0.913344,0.926754,0.920000,0.983610
2,Decision Tree,0.953638,0.933235,0.932626,0.932931,0.948684
3,XGBoost,0.962324,0.941624,0.949918,0.945753,0.993313
4,MLP,0.963245,0.949284,0.945392,0.947334,0.990993
5,Class-Weighted Random Forest,0.969323,0.952939,0.959659,0.956287,0.994557



✓ Table IV comparison saved: /content/drive/MyDrive/Phishing_Detection_Project_New/06_Results/final_comparison_all_6_main_models.csv


In [ ]:
# Show the exact 27 features used in the final dataset

feature_columns = [
    col for col in df.columns
    if col != "phishing"
]

print("Number of features:", len(feature_columns))

print("\n===== FINAL 27 FEATURES =====")
for i, feature in enumerate(feature_columns, start=1):
    print(f"{i}. {feature}")

Number of features: 27

===== FINAL 27 FEATURES =====
1. directory_length
2. time_domain_activation
3. qty_hyphen_directory
4. length_url
5. file_length
6. ttl_hostname
7. asn_ip
8. time_response
9. qty_percent_file
10. time_domain_expiration
11. domain_length
12. qty_hyphen_file
13. qty_underline_directory
14. qty_dot_domain
15. qty_vowels_domain
16. qty_nameservers
17. qty_mx_servers
18. qty_dot_url
19. qty_ip_resolved
20. qty_redirects
21. qty_hyphen_url
22. tls_ssl_certificate
23. qty_slash_url
24. qty_plus_directory
25. qty_asterisk_directory
26. qty_at_directory
27. qty_dot_file


## STEP 16 — Feature Ablation: 47 Features vs 27 Features


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# ==========================================
# Load cleaned 98-feature dataset
# ==========================================

data_path = f"{CLEAN_DIR}/cleaned_dataset.csv"

df_clean = pd.read_csv(data_path)

print("Cleaned dataset loaded successfully!")
print("Shape:", df_clean.shape)


# ==========================================
# Exact 47 features from original pipeline
# ==========================================

selected_47_features = [
    'directory_length',
    'time_domain_activation',
    'qty_hyphen_directory',
    'length_url',
    'file_length',
    'qty_slash_directory',
    'qty_at_file',
    'qty_slash_file',
    'qty_dot_directory',
    'qty_dot_file',
    'ttl_hostname',
    'asn_ip',
    'time_response',
    'qty_percent_file',
    'qty_equal_directory',
    'qty_space_file',
    'qty_asterisk_file',
    'qty_exclamation_directory',
    'time_domain_expiration',
    'domain_length',
    'qty_hyphen_file',
    'qty_underline_directory',
    'qty_slash_url',
    'qty_tilde_file',
    'qty_asterisk_directory',
    'qty_equal_file',
    'qty_plus_directory',
    'qty_dollar_directory',
    'qty_dollar_file',
    'qty_space_directory',
    'qty_dot_domain',
    'qty_vowels_domain',
    'qty_nameservers',
    'qty_mx_servers',
    'qty_and_file',
    'qty_comma_directory',
    'qty_plus_file',
    'qty_hashtag_directory',
    'qty_and_directory',
    'qty_comma_file',
    'qty_dot_url',
    'qty_ip_resolved',
    'qty_redirects',
    'qty_hyphen_url',
    'qty_questionmark_directory',
    'tls_ssl_certificate',
    'qty_at_directory'
]

print("Number of selected features:", len(selected_47_features))


# ==========================================
# Prepare data
# ==========================================

X_47 = df_clean[selected_47_features]
y = df_clean["phishing"]

X_train_47, X_test_47, y_train_47, y_test_47 = train_test_split(
    X_47,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)


# ==========================================
# Scaling
# ==========================================

scaler_47 = StandardScaler()

X_train_47_scaled = scaler_47.fit_transform(X_train_47)
X_test_47_scaled = scaler_47.transform(X_test_47)


# ==========================================
# Best MLP configuration
# ==========================================

mlp_47 = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    batch_size=32,
    learning_rate_init=0.001,
    max_iter=100,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    random_state=42
)

print("\nTraining MLP using 47 features...")

mlp_47.fit(
    X_train_47_scaled,
    y_train_47
)

print("Training completed!")
print("Iterations:", mlp_47.n_iter_)


# ==========================================
# Evaluation
# ==========================================

y_pred_47 = mlp_47.predict(X_test_47_scaled)
y_prob_47 = mlp_47.predict_proba(X_test_47_scaled)[:, 1]

accuracy_47 = accuracy_score(y_test_47, y_pred_47)
precision_47 = precision_score(y_test_47, y_pred_47)
recall_47 = recall_score(y_test_47, y_pred_47)
f1_47 = f1_score(y_test_47, y_pred_47)
roc_auc_47 = roc_auc_score(y_test_47, y_prob_47)

print("\n===== MLP WITH 47 FEATURES =====")

print(f"Accuracy:  {accuracy_47:.6f}")
print(f"Precision: {precision_47:.6f}")
print(f"Recall:    {recall_47:.6f}")
print(f"F1-score:  {f1_47:.6f}")
print(f"ROC-AUC:   {roc_auc_47:.6f}")

Cleaned dataset loaded successfully!
Shape: (87197, 99)
Number of selected features: 47

Training MLP using 47 features...
Training completed!
Iterations: 44

===== MLP WITH 47 FEATURES =====
Accuracy:  0.959232
Precision: 0.951399
Recall:    0.930961
F1-score:  0.941069
ROC-AUC:   0.990619


In [ ]:
import pandas as pd
import os

results_dir = RESULT_DIR
os.makedirs(results_dir, exist_ok=True)

ablation_results = pd.DataFrame({
    "Feature_Set": ["47 Features", "27 Features"],
    "Accuracy": [0.959232, 0.963245],
    "Precision": [0.951399, 0.949284],
    "Recall": [0.930961, 0.945392],
    "F1-score": [0.941069, 0.947334],
    "ROC-AUC": [0.990619, 0.990993]
})

ablation_results.to_csv(
    f"{results_dir}/mlp_feature_ablation_47_vs_27.csv",
    index=False
)

print("MLP feature ablation results saved successfully!")
print(ablation_results)

MLP feature ablation results saved successfully!
   Feature_Set  Accuracy  Precision    Recall  F1-score   ROC-AUC
0  47 Features  0.959232   0.951399  0.930961  0.941069  0.990619
1  27 Features  0.963245   0.949284  0.945392  0.947334  0.990993


## STEP 17 — Supplementary Deep Learning Models
These models are additional comparisons: TabNet, FT-Transformer, and Tabular ResNet.


In [ ]:
!pip -q install pytorch-tabnet

In [ ]:
# ============================================================
# TABNET 10-FOLD CV — CORRECTED VERSION WITH PROGRESS
# ============================================================

# Install if needed
!pip -q install pytorch-tabnet

# ------------------------------------------------------------
# Imports
# ------------------------------------------------------------

import os
import pickle
import time
import numpy as np
import pandas as pd
import torch

from pytorch_tabnet.tab_model import TabNetClassifier

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# ============================================================
# PATHS
# ============================================================

BASE_DIR = PROJECT_DIR
SAVED_DIR = os.path.join(BASE_DIR, "saved_state")

DATA_PATH = os.path.join(
    BASE_DIR,
    "phishing_27_features.csv"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "tabnet_results"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# LOAD FEATURE NAMES
# ============================================================

feature_file = os.path.join(
    SAVED_DIR,
    "feature_names.pkl"
)

with open(feature_file, "rb") as f:
    feature_names = pickle.load(f)

print("=" * 70)
print("FINAL FEATURE INFORMATION")
print("=" * 70)

print("Number of features:", len(feature_names))
print("Features:")
print(feature_names)

# ============================================================
# LOAD DATASET
# ============================================================

print("\nLoading dataset...")

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

# ============================================================
# CREATE X AND y
# ============================================================

X = df[feature_names].copy()
y = df["phishing"].copy()

print("\nX shape:", X.shape)
print("y shape:", y.shape)

print("\nClass Distribution:")
print(y.value_counts())

# ============================================================
# CHECK MISSING VALUES
# ============================================================

print("\nChecking missing values...")

print("Missing values in X:", X.isnull().sum().sum())
print("Missing values in y:", y.isnull().sum())

# Replace infinite values if any
X = X.replace([np.inf, -np.inf], np.nan)

# Fill missing values
X = X.fillna(X.median())

print("Data cleaning completed!")

# ============================================================
# 80:20 STRATIFIED TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\n" + "=" * 70)
print("TRAIN-TEST SPLIT")
print("=" * 70)

print("Training data:", X_train.shape)
print("Testing data :", X_test.shape)

# ============================================================
# CONVERT TO NUMPY
# ============================================================

X_train_np = X_train.values.astype(np.float32)
y_train_np = y_train.values.astype(np.int64)

print("\nNumPy conversion completed!")

# ============================================================
# HARDWARE INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("HARDWARE INFORMATION")
print("=" * 70)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU not available!")
    print("TabNet will run on CPU and may take a long time.")

# ============================================================
# 10-FOLD STRATIFIED CROSS VALIDATION
# ============================================================

skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

tabnet_fold_results = []

print("\n" + "=" * 70)
print("TABNET 10-FOLD STRATIFIED CROSS-VALIDATION")
print("=" * 70)

total_start_time = time.time()

# ============================================================
# FOLD LOOP
# ============================================================

for fold, (train_idx, val_idx) in enumerate(
    skf.split(X_train_np, y_train_np),
    start=1
):

    print("\n" + "=" * 70)
    print(f"STARTING FOLD {fold}/10")
    print("=" * 70)

    fold_start_time = time.time()

    # --------------------------------------------------------
    # CREATE FOLD DATA
    # --------------------------------------------------------

    X_tr = X_train_np[train_idx]
    X_val = X_train_np[val_idx]

    y_tr = y_train_np[train_idx]
    y_val = y_train_np[val_idx]

    print("Training samples  :", X_tr.shape)
    print("Validation samples:", X_val.shape)

    # --------------------------------------------------------
    # SCALE INSIDE FOLD
    # --------------------------------------------------------

    print("\nScaling data...")

    fold_scaler = StandardScaler()

    X_tr_scaled = fold_scaler.fit_transform(X_tr).astype(np.float32)
    X_val_scaled = fold_scaler.transform(X_val).astype(np.float32)

    print("Scaling completed!")

    # --------------------------------------------------------
    # CREATE TABNET MODEL
    # --------------------------------------------------------

    print("\nInitializing TabNet model...")

    model = TabNetClassifier(

        n_d=16,
        n_a=16,
        n_steps=3,

        gamma=1.5,
        lambda_sparse=1e-4,

        optimizer_fn=torch.optim.Adam,

        optimizer_params={
            "lr": 0.001
        },

        mask_type="entmax",

        seed=42 + fold,

        verbose=1,

        device_name="auto"
    )

    print("TabNet model initialized!")
    print("Starting training...")

    # --------------------------------------------------------
    # TRAIN MODEL
    # --------------------------------------------------------

    model.fit(

        X_tr_scaled,
        y_tr,

        eval_set=[
            (X_tr_scaled, y_tr),
            (X_val_scaled, y_val)
        ],

        eval_name=[
            "train",
            "validation"
        ],

        eval_metric=[
            "auc"
        ],

        max_epochs=50,

        patience=8,

        batch_size=1024,

        virtual_batch_size=128,

        num_workers=0,

        drop_last=False
    )

    print("\nTraining completed!")

    # --------------------------------------------------------
    # PREDICTION
    # --------------------------------------------------------

    print("Generating predictions...")

    y_pred = model.predict(
        X_val_scaled
    ).reshape(-1)

    y_prob = model.predict_proba(
        X_val_scaled
    )[:, 1]

    # --------------------------------------------------------
    # CALCULATE METRICS
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_val,
        y_pred
    )

    precision = precision_score(
        y_val,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        y_pred,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_val,
        y_prob
    )

    # --------------------------------------------------------
    # FOLD TIME
    # --------------------------------------------------------

    fold_time = time.time() - fold_start_time

    print("\n" + "-" * 70)
    print(f"FOLD {fold}/10 RESULTS")
    print("-" * 70)

    print(f"Accuracy : {accuracy:.6f}")
    print(f"Precision: {precision:.6f}")
    print(f"Recall   : {recall:.6f}")
    print(f"F1-score : {f1:.6f}")
    print(f"ROC-AUC  : {roc_auc:.6f}")
    print(f"Time     : {fold_time/60:.2f} minutes")

    # --------------------------------------------------------
    # SAVE FOLD RESULT
    # --------------------------------------------------------

    tabnet_fold_results.append({

        "Fold": fold,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1-score": f1,

        "ROC-AUC": roc_auc,

        "Training_Time_Seconds": fold_time
    })

# ============================================================
# CREATE RESULTS DATAFRAME
# ============================================================

tabnet_cv_df = pd.DataFrame(
    tabnet_fold_results
)

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("TABNET FINAL 10-FOLD CV SUMMARY")
print("=" * 70)

for metric in [

    "Accuracy",

    "Precision",

    "Recall",

    "F1-score",

    "ROC-AUC"

]:

    mean_value = tabnet_cv_df[metric].mean()

    std_value = tabnet_cv_df[metric].std()

    print(
        f"{metric:12s}: "
        f"{mean_value:.6f} ± {std_value:.6f}"
    )

# ============================================================
# TOTAL TIME
# ============================================================

total_time = time.time() - total_start_time

print("\n" + "=" * 70)
print("TOTAL TRAINING TIME")
print("=" * 70)

print(f"Total time: {total_time/60:.2f} minutes")

# ============================================================
# SAVE FOLD RESULTS
# ============================================================

fold_path = os.path.join(
    OUTPUT_DIR,
    "tabnet_cv_fold_results.csv"
)

tabnet_cv_df.to_csv(
    fold_path,
    index=False
)

# ============================================================
# CREATE SUMMARY DATAFRAME
# ============================================================

summary_df = pd.DataFrame({

    "Model": ["TabNet"],

    "Accuracy_Mean": [
        tabnet_cv_df["Accuracy"].mean()
    ],

    "Accuracy_Std": [
        tabnet_cv_df["Accuracy"].std()
    ],

    "Precision_Mean": [
        tabnet_cv_df["Precision"].mean()
    ],

    "Precision_Std": [
        tabnet_cv_df["Precision"].std()
    ],

    "Recall_Mean": [
        tabnet_cv_df["Recall"].mean()
    ],

    "Recall_Std": [
        tabnet_cv_df["Recall"].std()
    ],

    "F1_Mean": [
        tabnet_cv_df["F1-score"].mean()
    ],

    "F1_Std": [
        tabnet_cv_df["F1-score"].std()
    ],

    "ROC_AUC_Mean": [
        tabnet_cv_df["ROC-AUC"].mean()
    ],

    "ROC_AUC_Std": [
        tabnet_cv_df["ROC-AUC"].std()
    ]

})

# ============================================================
# SAVE SUMMARY
# ============================================================

summary_path = os.path.join(
    OUTPUT_DIR,
    "tabnet_cv_summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False
)

# ============================================================
# FINAL OUTPUT
# ============================================================

print("\n" + "=" * 70)
print("TABNET CV RESULTS SAVED SUCCESSFULLY")
print("=" * 70)

print("\nFold Results:")
print(fold_path)

print("\nSummary Results:")
print(summary_path)

print("\nFinal Summary DataFrame:")
print(summary_df)

In [ ]:
!pip -q install pytorch-tabular

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.8/165.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 9.4 MB/s eta 0:00:00


In [ ]:
# ============================================================
# FT-TRANSFORMER — 10-FOLD STRATIFIED CROSS-VALIDATION
# ============================================================

import os
import pickle
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# ------------------------------------------------------------
# ------------------------------------------------------------

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

BASE_DIR = PROJECT_DIR
SAVED_DIR = os.path.join(BASE_DIR, "saved_state")
FEATURE_DIR = os.path.join(BASE_DIR, "03_Feature_Selection") # Define FEATURE_DIR

# IMPORTANT:
# Change this only if your actual 27-feature dataset has another name.
DATA_PATH = os.path.join(
    FEATURE_DIR,
    "phishing_27_features.csv"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "ft_transformer_results"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Load final feature names
# ------------------------------------------------------------

# Original code attempted to load from file, but `reduced_features` is already available.
# Replaced file loading with direct assignment to prevent FileNotFoundError.
feature_names = reduced_features

print("=" * 70)
print("FINAL FEATURE INFORMATION")
print("=" * 70)

print("Number of features:", len(feature_names))
print("Features:", feature_names)

# ------------------------------------------------------------
# Load dataset
# ------------------------------------------------------------

df = pd.read_csv(DATA_PATH)

print("\nDataset shape:", df.shape)

# ------------------------------------------------------------
# Identify target
# ------------------------------------------------------------

target_candidates = [
    "phishing",
    "label",
    "target",
    "class"
]

target_column = None

for col in target_candidates:
    if col in df.columns:
        target_column = col
        break

if target_column is None:
    raise ValueError(
        "Target column not found. "
        "Expected one of: phishing, label, target, class"
    )

print("Target column:", target_column)

# ------------------------------------------------------------
# X and y
# ------------------------------------------------------------

X = df[feature_names].copy()
y = df[target_column].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

# ------------------------------------------------------------
# Ensure numeric data
# ------------------------------------------------------------

X = X.replace([np.inf, -np.inf], np.nan)

if X.isnull().sum().sum() > 0:
    print("\nMissing values detected.")
    print("Applying median imputation...")

    X = X.fillna(X.median(numeric_only=True))

# ------------------------------------------------------------
# 80:20 stratified split
# Same random state used in previous experiment
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\n" + "=" * 70)
print("TRAIN-TEST SPLIT")
print("=" * 70)

print("Training data:", X_train.shape)
print("Testing data :", X_test.shape)

# ------------------------------------------------------------
# Convert to numpy
# ------------------------------------------------------------

X_train_np = X_train.values.astype(np.float32)
y_train_np = y_train.values.astype(np.int64)

# ------------------------------------------------------------
# Hardware
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("HARDWARE")
print("=" * 70)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# IMPORTANT
# ============================================================
#
# We will use PyTorch directly for FT-Transformer.
# This avoids depending on a specific pytorch-tabular version.
#
# ============================================================

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader


# ============================================================
# FT-TRANSFORMER MODEL
# ============================================================

class NumericalFeatureTokenizer(nn.Module):

    def __init__(self, num_features, d_token):
        super().__init__()

        self.weight = nn.Parameter(
            torch.randn(num_features, d_token) * 0.02
        )

        self.bias = nn.Parameter(
            torch.zeros(num_features, d_token)
        )

    def forward(self, x):

        # x:
        # [batch, features]

        x = x.unsqueeze(-1)

        # [batch, features, 1]

        tokens = (
            x * self.weight.unsqueeze(0)
            + self.bias.unsqueeze(0)
        )

        # [batch, features, d_token]

        return tokens


class FTTransformer(
nn.Module
):

    def __init__(
        self,
        num_features,
        d_token=32,
        n_heads=4,
        n_layers=2,
        dropout=0.1
    ):

        super().__init__()

        self.tokenizer = NumericalFeatureTokenizer(
            num_features,
            d_token
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token,
            nhead=n_heads,
            dim_feedforward=d_token * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers
        )

        self.norm = nn.LayerNorm(d_token)

        self.head = nn.Sequential(
            nn.Linear(d_token, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x):

        tokens = self.tokenizer(x)

        # Transformer processing
        tokens = self.transformer(tokens)

        # Mean pooling across feature tokens
        pooled = tokens.mean(dim=1)

        pooled = self.norm(pooled)

        output = self.head(pooled)

        return output.squeeze(1)


# ============================================================
# 10-FOLD STRATIFIED CROSS-VALIDATION
# ============================================================

skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

ft_fold_results = []

print("\n" + "=" * 70)
print("FT-TRANSFORMER 10-FOLD STRATIFIED CROSS-VALIDATION")
print("=" * 70)


# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


# ============================================================
# FOLD LOOP
# ============================================================

for fold, (train_idx, val_idx) in enumerate(
    skf.split(X_train_np, y_train_np),
    1
):

    print("\n" + "-" * 70)
    print(f"FOLD {fold}/10")
    print("-" * 70)

    # --------------------------------------------------------
    # Fold data
    # --------------------------------------------------------

    X_tr = X_train_np[train_idx]
    X_val = X_train_np[val_idx]

    y_tr = y_train_np[train_idx]
    y_val = y_train_np[val_idx]

    # --------------------------------------------------------
    # Scaling INSIDE fold
    # --------------------------------------------------------

    fold_scaler = StandardScaler()

    X_tr_scaled = fold_scaler.fit_transform(X_tr)
    X_val_scaled = fold_scaler.transform(X_val)

    # --------------------------------------------------------
    # Convert to tensors
    # --------------------------------------------------------

    X_tr_tensor = torch.tensor(
        X_tr_scaled,
        dtype=torch.float32
    )

    y_tr_tensor = torch.tensor(
        y_tr,
        dtype=torch.float32
    )

    X_val_tensor = torch.tensor(
        X_val_scaled,
        dtype=torch.float32
    )

    # --------------------------------------------------------
    # DataLoader
    # --------------------------------------------------------

    train_dataset = TensorDataset(
        X_tr_tensor,
        y_tr_tensor
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=1024,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    # --------------------------------------------------------
    # Create model
    # --------------------------------------------------------

    torch.manual_seed(42 + fold)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42 + fold)

    model = FTTransformer(
        num_features=X_train_np.shape[1],
        d_token=32,
        n_heads=4,
        n_layers=2,
        dropout=0.1
    ).to(device)

    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=0.001,
        weight_decay=1e-4
    )

    # --------------------------------------------------------
    # Loss
    # --------------------------------------------------------

    criterion = nn.BCEWithLogitsLoss()

    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------

    max_epochs = 40
    patience = 6

    best_auc = -np.inf
    best_state = None
    patience_counter = 0

    # ========================================================
    # TRAINING
    # ========================================================

    for epoch in range(1, max_epochs + 1):

        model.train()

        total_loss = 0.0
        total_samples = 0

        for batch_X, batch_y in train_loader:

            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()

            logits = model(batch_X)

            loss = criterion(
                logits,
                batch_y
            )

            loss.backward()

            optimizer.step()

            total_loss += (
                loss.item() * len(batch_X)
            )

            total_samples += len(batch_X)

        avg_loss = (
            total_loss / total_samples
        )

        # ----------------------------------------------------
        # Validation
        # ----------------------------------------------------

        model.eval()

        with torch.no_grad():

            val_logits = model(
                X_val_tensor.to(device)
            )

            val_prob = torch.sigmoid(
                val_logits
            ).cpu().numpy()

        val_pred = (
            val_prob >= 0.5
        ).astype(int)

        val_auc = roc_auc_score(
            y_val,
            val_prob
        )

        # ----------------------------------------------------
        # Early stopping
        # ----------------------------------------------------

        if val_auc > best_auc:

            best_auc = val_auc

            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

            patience_counter = 0

        else:

            patience_counter += 1

        if epoch == 1 or epoch % 5 == 0:

            print(
                f"Epoch {epoch:02d} | "
                f"Loss={avg_loss:.4f} | "
                f"Val-AUC={val_auc:.4f}"
            )

        if patience_counter >= patience:

            print(
                f"Early stopping at epoch {epoch}"
            )

            break

    # --------------------------------------------------------
    # Restore best model
    # --------------------------------------------------------

    model.load_state_dict(
        best_state
    )

    model.eval()

    # --------------------------------------------------------
    # Final validation prediction
    # --------------------------------------------------------

    with torch.no_grad():

        val_logits = model(
            X_val_tensor.to(device)
        )

        y_prob = torch.sigmoid(
            val_logits
        ).cpu().numpy()

    y_pred = (
        y_prob >= 0.5
    ).astype(int)

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_val,
        y_pred
    )

    precision = precision_score(
        y_val,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        y_pred,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_val,
        y_prob
    )

    print(
        f"Fold {fold:2d} | "
        f"Accuracy={accuracy:.4f} | "
        f"Precision={precision:.4f} | "
        f"Recall={recall:.4f} | "
        f"F1={f1:.4f} | "
        f"ROC-AUC={roc_auc:.4f}"
    )

    ft_fold_results.append({

        "Fold": fold,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1-score": f1,

        "ROC-AUC": roc_auc
    })


# ============================================================
# FINAL SUMMARY
# ============================================================

ft_cv_df = pd.DataFrame(
    ft_fold_results
)

print("\n" + "=" * 70)
print("FT-TRANSFORMER FINAL 10-FOLD CV SUMMARY")
print("=" * 70)

for metric in [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "ROC-AUC"
]:

    mean_value = ft_cv_df[metric].mean()

    std_value = ft_cv_df[metric].std()

    print(
        f"{metric:10s}: "
        f"{mean_value:.6f} \u00b1 {std_value:.6f}"
    )


# ============================================================
# SAVE RESULTS
# ============================================================

fold_path = os.path.join(
    OUTPUT_DIR,
    "ft_transformer_cv_fold_results.csv"
)

summary_path = os.path.join(
    OUTPUT_DIR,
    "ft_transformer_cv_summary.csv"
)

ft_cv_df.to_csv(
    fold_path,
    index=False
)

summary_df = pd.DataFrame({

    "Model": ["FT-Transformer"],

    "Accuracy_Mean": [
        ft_cv_df["Accuracy"].mean()
    ],

    "Accuracy_Std": [
        ft_cv_df["Accuracy"].std()
    ],

    "Precision_Mean": [
        ft_cv_df["Precision"].mean()
    ],

    "Precision_Std": [
        ft_cv_df["Precision"].std()
    ],

    "Recall_Mean": [
        ft_cv_df["Recall"].mean()
    ],

    "Recall_Std": [
        ft_cv_df["Recall"].std()
    ],

    "F1_Mean": [
        ft_cv_df["F1-score"].mean()
    ],

    "F1_Std": [
        ft_cv_df["F1-score"].std()
    ],

    "ROC_AUC_Mean": [
        ft_cv_df["ROC-AUC"].mean()
    ],

    "ROC_AUC_Std": [
        ft_cv_df["ROC-AUC"].std()
    ]
})

summary_df.to_csv(
    summary_path,
    index=False
)

print("\n" + "=" * 70)
print("FT-TRANSFORMER RESULTS SAVED")
print("=" * 70)

print("\u2713", fold_path)
print("\u2713", summary_path)


FINAL FEATURE INFORMATION
Number of features: 27
Features: ['directory_length', 'time_domain_activation', 'qty_hyphen_directory', 'length_url', 'file_length', 'ttl_hostname', 'asn_ip', 'time_response', 'qty_percent_file', 'time_domain_expiration', 'domain_length', 'qty_hyphen_file', 'qty_underline_directory', 'qty_dot_domain', 'qty_vowels_domain', 'qty_nameservers', 'qty_mx_servers', 'qty_dot_url', 'qty_ip_resolved', 'qty_redirects', 'qty_hyphen_url', 'tls_ssl_certificate', 'qty_slash_url', 'qty_plus_directory', 'qty_asterisk_directory', 'qty_at_directory', 'qty_dot_file']

Dataset shape: (87197, 28)
Target column: phishing
X shape: (87197, 27)
y shape: (87197,)

TRAIN-TEST SPLIT
Training data: (69757, 27)
Testing data : (17440, 27)

HARDWARE
CUDA available: False

FT-TRANSFORMER 10-FOLD STRATIFIED CROSS-VALIDATION
Using device: cpu

----------------------------------------------------------------------
FOLD 1/10
----------------------------------------------------------------------


/tmp/ipykernel_4969/763749820.py:237: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Epoch 01 | Loss=0.2711 | Val-AUC=0.9774
Epoch 05 | Loss=0.1316 | Val-AUC=0.9887
Epoch 10 | Loss=0.1178 | Val-AUC=0.9885
Epoch 15 | Loss=0.1132 | Val-AUC=0.9906
Epoch 20 | Loss=0.1054 | Val-AUC=0.9913
Epoch 25 | Loss=0.1038 | Val-AUC=0.9914
Epoch 30 | Loss=0.1001 | Val-AUC=0.9911
Epoch 35 | Loss=0.0956 | Val-AUC=0.9920
Epoch 40 | Loss=0.0943 | Val-AUC=0.9922
Fold  1 | Accuracy=0.9590 | Precision=0.9378 | Recall=0.9455 | F1=0.9416 | ROC-AUC=0.9922

----------------------------------------------------------------------
FOLD 2/10
----------------------------------------------------------------------


/tmp/ipykernel_4969/763749820.py:237: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Epoch 01 | Loss=0.2689 | Val-AUC=0.9775
Epoch 05 | Loss=0.1350 | Val-AUC=0.9886
Epoch 10 | Loss=0.1198 | Val-AUC=0.9904
Epoch 15 | Loss=0.1108 | Val-AUC=0.9906


In [ ]:
# ================================================================
# TABULAR RESNET
# FINAL 27 FEATURES
# 10-FOLD STRATIFIED CROSS-VALIDATION
# FULL CORRECTED VERSION
# ================================================================

import os
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

import torch
import torch.nn as nn

from torch.utils.data import (
    TensorDataset,
    DataLoader
)


# ================================================================
# 1. PROJECT PATH
# ================================================================

PROJECT_DIR = (
    "/content/gdrive2/MyDrive/"
    "Phishing_Detection_Project"
)

DATA_PATH = os.path.join(
    PROJECT_DIR,
    "phishing_27_features.csv"
)

RESULT_DIR = os.path.join(
    PROJECT_DIR,
    "tabular_resnet_results"
)

os.makedirs(
    RESULT_DIR,
    exist_ok=True
)


print("=" * 70)
print("TABULAR RESNET - 10-FOLD STRATIFIED CROSS-VALIDATION")
print("=" * 70)


# ================================================================
# 2. CHECK DATASET
# ================================================================

if not os.path.exists(DATA_PATH):

    raise FileNotFoundError(
        f"\nDataset not found:\n{DATA_PATH}"
    )

print("✓ Dataset found")


# ================================================================
# 3. LOAD DATASET
# ================================================================

df = pd.read_csv(
    DATA_PATH
)

print(
    "\nDataset shape:",
    df.shape
)


# ================================================================
# 4. TARGET COLUMN
# ================================================================

TARGET_COLUMN = "phishing"

if TARGET_COLUMN not in df.columns:

    raise ValueError(
        f"Target column '{TARGET_COLUMN}' "
        f"not found in dataset."
    )


# ================================================================
# 5. AUTOMATICALLY GET 27 FEATURES
# ================================================================
#
# Dataset structure:
#
# 27 selected features
# +
# 1 target = phishing
# =
# 28 columns
#
# Therefore, simply remove the target column.
#
# We intentionally DO NOT read final_27_features.txt
# to avoid formatting/header issues.
# ================================================================

feature_names = [
    col
    for col in df.columns
    if col != TARGET_COLUMN
]


print(
    "\nNumber of input features:",
    len(feature_names)
)


if len(feature_names) != 27:

    raise ValueError(
        f"Expected exactly 27 features, "
        f"but found {len(feature_names)}."
    )


print("\n✓ Exactly 27 features identified")


print("\nSelected features:")

for i, feature in enumerate(
    feature_names,
    start=1
):

    print(
        f"{i:02d}. {feature}"
    )


# ================================================================
# 6. CREATE X AND y
# ================================================================

X = df[
    feature_names
].copy()

y = df[
    TARGET_COLUMN
].copy()


# ================================================================
# 7. CONVERT FEATURES TO NUMERIC
# ================================================================

for col in feature_names:

    X[col] = pd.to_numeric(
        X[col],
        errors="coerce"
    )


# ================================================================
# 8. HANDLE INFINITY
# ================================================================

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)


# ================================================================
# 9. TARGET CONVERSION
# ================================================================

y = pd.to_numeric(
    y,
    errors="raise"
).astype(
    np.int64
)


# ================================================================
# 10. BASIC DATA INFORMATION
# ================================================================

print("\n" + "=" * 70)
print("DATA INFORMATION")
print("=" * 70)

print(
    "Samples        :",
    len(X)
)

print(
    "Features       :",
    X.shape[1]
)

print(
    "Target         :",
    TARGET_COLUMN
)

print(
    "Missing values :",
    X.isna().sum().sum()
)

print(
    "Infinite values:",
    np.isinf(
        X.select_dtypes(
            include=np.number
        ).values
    ).sum()
)


print("\nClass distribution:")

print(
    y.value_counts()
    .sort_index()
)


# ================================================================
# 11. CONVERT TO NUMPY
# ================================================================

X_np = X.values.astype(
    np.float32
)

y_np = y.values.astype(
    np.int64
)


# ================================================================
# 12. GPU / DEVICE CHECK
# ================================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("\n" + "=" * 70)
print("DEVICE")
print("=" * 70)

print(
    "PyTorch:",
    torch.__version__
)

print(
    "Device:",
    device
)


if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# ================================================================
# 13. RESIDUAL BLOCK
# ================================================================

class ResidualBlock(
    nn.Module
):

    def __init__(
        self,
        hidden_dim=128,
        dropout=0.20
    ):

        super().__init__()


        self.fc1 = nn.Linear(
            hidden_dim,
            hidden_dim
        )


        self.bn1 = nn.BatchNorm1d(
            hidden_dim
        )


        self.fc2 = nn.Linear(
            hidden_dim,
            hidden_dim
        )


        self.bn2 = nn.BatchNorm1d(
            hidden_dim
        )


        self.relu = nn.ReLU()


        self.dropout = nn.Dropout(
            dropout
        )


    def forward(
        self,
        x
    ):

        residual = x


        x = self.fc1(
            x
        )


        x = self.bn1(
            x
        )


        x = self.relu(
            x
        )


        x = self.dropout(
            x
        )


        x = self.fc2(
            x
        )


        x = self.bn2(
            x
        )


        x = x + residual


        x = self.relu(
            x
        )


        return x


# ================================================================
# 14. TABULAR RESNET MODEL
# ================================================================

class TabularResNet(
    nn.Module
):

    def __init__(
        self,
        input_dim=27,
        hidden_dim=128,
        num_blocks=3,
        dropout=0.20
    ):

        super().__init__()


        self.input_layer = nn.Sequential(

            nn.Linear(
                input_dim,
                hidden_dim
            ),

            nn.BatchNorm1d(
                hidden_dim
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            )
        )


        self.residual_layers = nn.Sequential(

            *[
                ResidualBlock(
                    hidden_dim=hidden_dim,
                    dropout=dropout
                )

                for _ in range(num_blocks)
            ]
        )


        self.output_layer = nn.Linear(
            hidden_dim,
            1
        )


    def forward(
        self,
        x
    ):

        x = self.input_layer(
            x
        )


        x = self.residual_layers(
            x
        )


        x = self.output_layer(
            x
        )


        return x.squeeze(1)


# ================================================================
# 15. TRAIN ONE FOLD
# ================================================================

def train_one_fold(
    X_train,
    y_train,
    X_val,
    y_val,
    fold_number
):


    # ------------------------------------------------------------
    # MISSING VALUE IMPUTATION
    # ------------------------------------------------------------
    #
    # IMPORTANT:
    # Median is calculated ONLY from training fold.
    #
    # This prevents validation information leakage.
    # ------------------------------------------------------------

    train_medians = np.nanmedian(
        X_train,
        axis=0
    )


    train_medians = np.where(
        np.isnan(train_medians),
        0,
        train_medians
    )


    X_train = np.where(
        np.isnan(X_train),
        train_medians,
        X_train
    )


    X_val = np.where(
        np.isnan(X_val),
        train_medians,
        X_val
    )


    # ------------------------------------------------------------
    # STANDARDIZATION
    # ------------------------------------------------------------

    scaler = StandardScaler()


    X_train_scaled = scaler.fit_transform(
        X_train
    )


    X_val_scaled = scaler.transform(
        X_val
    )


    # ------------------------------------------------------------
    # TORCH TENSORS
    # ------------------------------------------------------------

    X_train_tensor = torch.tensor(
        X_train_scaled,
        dtype=torch.float32
    )


    y_train_tensor = torch.tensor(
        y_train,
        dtype=torch.float32
    )


    X_val_tensor = torch.tensor(
        X_val_scaled,
        dtype=torch.float32
    )


    # ------------------------------------------------------------
    # DATASET / DATALOADER
    # ------------------------------------------------------------

    train_dataset = TensorDataset(
        X_train_tensor,
        y_train_tensor
    )


    train_loader = DataLoader(
        train_dataset,
        batch_size=512,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )


    # ------------------------------------------------------------
    # MODEL
    # ------------------------------------------------------------

    model = TabularResNet(
        input_dim=27,
        hidden_dim=128,
        num_blocks=3,
        dropout=0.20
    ).to(
        device
    )


    # ------------------------------------------------------------
    # LOSS FUNCTION
    # ------------------------------------------------------------

    criterion = nn.BCEWithLogitsLoss()


    # ------------------------------------------------------------
    # OPTIMIZER
    # ------------------------------------------------------------

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=0.001,
        weight_decay=1e-4
    )


    # ------------------------------------------------------------
    # LEARNING RATE SCHEDULER
    # ------------------------------------------------------------

    scheduler = (
        torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=4
        )
    )


    # ------------------------------------------------------------
    # TRAINING SETTINGS
    # ------------------------------------------------------------

    MAX_EPOCHS = 50

    PATIENCE = 8

    best_auc = -np.inf

    best_state = None

    patience_counter = 0


    # ============================================================
    # TRAINING LOOP
    # ============================================================

    for epoch in range(
        1,
        MAX_EPOCHS + 1
    ):


        model.train()


        total_loss = 0.0


        # --------------------------------------------------------
        # MINI-BATCH TRAINING
        # --------------------------------------------------------

        for (
            batch_X,
            batch_y
        ) in train_loader:


            batch_X = batch_X.to(
                device,
                non_blocking=True
            )


            batch_y = batch_y.to(
                device,
                non_blocking=True
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            logits = model(
                batch_X
            )


            loss = criterion(
                logits,
                batch_y
            )


            loss.backward()


            optimizer.step()


            total_loss += (
                loss.item()
                * batch_X.size(0)
            )


        # --------------------------------------------------------
        # AVERAGE TRAINING LOSS
        # --------------------------------------------------------

        avg_loss = (
            total_loss
            / len(train_dataset)
        )


        # --------------------------------------------------------
        # VALIDATION
        # --------------------------------------------------------

        model.eval()


        with torch.no_grad():

            val_logits = model(
                X_val_tensor.to(
                    device,
                    non_blocking=True
                )
            )


            val_prob = torch.sigmoid(
                val_logits
            ).cpu().numpy()


        val_auc = roc_auc_score(
            y_val,
            val_prob
        )


        # --------------------------------------------------------
        # LEARNING RATE SCHEDULER
        # --------------------------------------------------------

        scheduler.step(
            val_auc
        )


        # --------------------------------------------------------
        # PRINT PROGRESS
        # --------------------------------------------------------

        if (
            epoch == 1
            or epoch % 5 == 0
        ):

            print(
                f"Epoch {epoch:02d} | "
                f"Loss={avg_loss:.4f} | "
                f"Val-AUC={val_auc:.4f}"
            )


        # --------------------------------------------------------
        # SAVE BEST MODEL
        # --------------------------------------------------------

        if val_auc > best_auc:

            best_auc = val_auc


            best_state = {
                key: value.detach()
                .cpu()
                .clone()

                for key, value
                in model.state_dict().items()
            }


            patience_counter = 0


        else:

            patience_counter += 1


        # --------------------------------------------------------
        # EARLY STOPPING
        # --------------------------------------------------------

        if (
            patience_counter
            >= PATIENCE
        ):

            print(
                f"Early stopping at epoch "
                f"{epoch}"
            )

            break


    # ============================================================
    # RESTORE BEST MODEL
    # ============================================================

    model.load_state_dict(
        best_state
    )


    model.eval()


    # ============================================================
    # FINAL VALIDATION PREDICTION
    # ============================================================

    with torch.no_grad():

        val_logits = model(
            X_val_tensor.to(
                device,
                non_blocking=True
            )
        )


        val_prob = torch.sigmoid(
            val_logits
        ).cpu().numpy()


    # ------------------------------------------------------------
    # CLASSIFICATION THRESHOLD
    # ------------------------------------------------------------

    val_pred = (
        val_prob >= 0.5
    ).astype(
        int
    )


    # ============================================================
    # CALCULATE METRICS
    # ============================================================

    accuracy = accuracy_score(
        y_val,
        val_pred
    )


    precision = precision_score(
        y_val,
        val_pred,
        zero_division=0
    )


    recall = recall_score(
        y_val,
        val_pred,
        zero_division=0
    )


    f1 = f1_score(
        y_val,
        val_pred,
        zero_division=0
    )


    auc = roc_auc_score(
        y_val,
        val_prob
    )


    # ============================================================
    # FOLD RESULT
    # ============================================================

    print(
        f"Fold {fold_number:2d} | "
        f"Accuracy={accuracy:.4f} | "
        f"Precision={precision:.4f} | "
        f"Recall={recall:.4f} | "
        f"F1={f1:.4f} | "
        f"ROC-AUC={auc:.4f}"
    )


    return {

        "Fold": fold_number,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1-score": f1,

        "ROC-AUC": auc
    }


# ================================================================
# 16. 10-FOLD STRATIFIED CROSS-VALIDATION
# ================================================================

print("\n")

print("=" * 70)

print(
    "STARTING 10-FOLD STRATIFIED CROSS-VALIDATION"
)

print("=" * 70)


skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)


fold_results = []


for fold_number, (
    train_idx,
    val_idx
) in enumerate(
    skf.split(
        X_np,
        y_np
    ),
    start=1
):


    print("\n")

    print(
        "-" * 70
    )

    print(
        f"FOLD {fold_number}/10"
    )

    print(
        "-" * 70
    )


    X_train = X_np[
        train_idx
    ]


    y_train = y_np[
        train_idx
    ]


    X_val = X_np[
        val_idx
    ]


    y_val = y_np[
        val_idx
    ]


    result = train_one_fold(

        X_train=X_train,

        y_train=y_train,

        X_val=X_val,

        y_val=y_val,

        fold_number=fold_number
    )


    fold_results.append(
        result
    )


# ================================================================
# 17. CREATE FOLD RESULTS DATAFRAME
# ================================================================

resnet_cv_df = pd.DataFrame(
    fold_results
)


# ================================================================
# 18. CALCULATE MEAN AND STANDARD DEVIATION
# ================================================================

print("\n")

print("=" * 70)

print(
    "TABULAR RESNET FINAL 10-FOLD CV SUMMARY"
)

print("=" * 70)


summary_rows = {}


for metric in [

    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "ROC-AUC"

]:


    mean_value = (
        resnet_cv_df[
            metric
        ].mean()
    )


    std_value = (
        resnet_cv_df[
            metric
        ].std()
    )


    summary_rows[
        metric + "_Mean"
    ] = mean_value


    summary_rows[
        metric + "_Std"
    ] = std_value


    print(
        f"{metric:10s}: "
        f"{mean_value:.6f} ± "
        f"{std_value:.6f}"
    )


# ================================================================
# 19. SUMMARY DATAFRAME
# ================================================================

resnet_summary_df = pd.DataFrame(
    [summary_rows]
)


# ================================================================
# 20. SAVE FOLD RESULTS
# ================================================================

fold_results_path = os.path.join(
    RESULT_DIR,
    "resnet_cv_fold_results.csv"
)


resnet_cv_df.to_csv(
    fold_results_path,
    index=False
)


# ================================================================
# 21. SAVE SUMMARY
# ================================================================

summary_results_path = os.path.join(
    RESULT_DIR,
    "resnet_cv_summary.csv"
)


resnet_summary_df.to_csv(
    summary_results_path,
    index=False
)


# ================================================================
# 22. SAVE FEATURE LIST USED BY RESNET
# ================================================================

resnet_feature_path = os.path.join(
    RESULT_DIR,
    "resnet_27_features_used.txt"
)


with open(
    resnet_feature_path,
    "w"
) as f:

    for feature in feature_names:

        f.write(
            feature + "\n"
        )


# ================================================================
# 23. FINAL OUTPUT
# ================================================================

print("\n")

print("=" * 70)

print(
    "TABULAR RESNET RESULTS SAVED"
)

print("=" * 70)


print(
    "✓ Fold results:"
)

print(
    fold_results_path
)


print(
    "\n✓ Summary:"
)

print(
    summary_results_path
)


print(
    "\n✓ Features used:"
)

print(
    resnet_feature_path
)


# ================================================================
# 24. DISPLAY RESULTS
# ================================================================

print("\n")

print("=" * 70)

print(
    "FOLD-WISE RESULTS"
)

print("=" * 70)

display(
    resnet_cv_df
)


print("\n")

print("=" * 70)

print(
    "FINAL SUMMARY"
)

print("=" * 70)

display(
    resnet_summary_df
)


print("\n")

print("=" * 70)

print(
    "TABULAR RESNET 10-FOLD CV COMPLETED SUCCESSFULLY"
)

print("=" * 70)

## STEP 18 — Exploratory Hybrid Model: MLP + Tabular ResNet


In [ ]:
# ============================================================
# HYBRID DL MODEL - STEP 1
# Check Existing Project Files
# ============================================================

import os

BASE_DIR = PROJECT_DIR

print("=" * 70)
print("PROJECT DIRECTORY CHECK")
print("=" * 70)

print("BASE_DIR:", BASE_DIR)
print("Exists:", os.path.exists(BASE_DIR))

if os.path.exists(BASE_DIR):
    print("\nProject folders/files:")
    for item in sorted(os.listdir(BASE_DIR)):
        print(" -", item)

print("\n" + "=" * 70)
print("CHECKING SAVED STATE")
print("=" * 70)

SAVED_STATE = os.path.join(BASE_DIR, "saved_state")

if os.path.exists(SAVED_STATE):
    for item in sorted(os.listdir(SAVED_STATE)):
        print(" -", item)
else:
    print("saved_state folder not found!")

print("\n" + "=" * 70)
print("CHECKING IMPORTANT FILES")
print("=" * 70)

important_files = [
    "feature_columns.pkl",
    "feature_names.pkl",
    "final_mlp.pkl",
    "scaler.pkl",
    "metrics.csv",
    "all_models_cv_fold_results.csv",
    "all_models_cv_summary.csv"
]

for file in important_files:
    path = os.path.join(SAVED_STATE, file)
    print(("✓" if os.path.exists(path) else "✗"), file)

In [ ]:
# ============================================================
# HYBRID DL MODEL - STEP 2 (CORRECTED)
# Dataset + 27 Features Verification
# ============================================================

import os
import pickle
import pandas as pd
import numpy as np

BASE_DIR = PROJECT_DIR
SAVED_STATE = os.path.join(BASE_DIR, "saved_state")

DATA_PATH = os.path.join(BASE_DIR, "phishing_27_features.csv")
FEATURE_PATH = os.path.join(SAVED_STATE, "feature_columns.pkl")
FEATURE_NAMES_PATH = os.path.join(SAVED_STATE, "feature_names.pkl")

# ------------------------------------------------------------
# 1. Load dataset
# ------------------------------------------------------------

df = pd.read_csv(DATA_PATH)

print("=" * 70)
print("DATASET CHECK")
print("=" * 70)

print("Dataset shape:", df.shape)

# ------------------------------------------------------------
# 2. Load feature information
# ------------------------------------------------------------

with open(FEATURE_PATH, "rb") as f:
    feature_columns = pickle.load(f)

with open(FEATURE_NAMES_PATH, "rb") as f:
    feature_names = pickle.load(f)

print("\n" + "=" * 70)
print("SAVED FEATURE INFORMATION")
print("=" * 70)

print("Number of feature columns:", len(feature_columns))
print("Number of feature names:", len(feature_names))

print("\nFeatures:")

for i, col in enumerate(feature_columns, 1):
    print(f"{i:2d}. {col}")

# ------------------------------------------------------------
# 3. Target
# ------------------------------------------------------------

TARGET_COLUMN = "phishing"

if TARGET_COLUMN not in df.columns:
    raise ValueError("Target column 'phishing' not found!")

# ------------------------------------------------------------
# 4. Feature consistency
# ------------------------------------------------------------

missing_features = [
    col for col in feature_columns
    if col not in df.columns
]

if missing_features:
    raise ValueError(
        f"Missing features: {missing_features}"
    )

print("\n" + "=" * 70)
print("FEATURE CONSISTENCY")
print("=" * 70)

print("✓ All saved features exist")
print("✓ Target column:", TARGET_COLUMN)

# ------------------------------------------------------------
# 5. Build X and y
# ------------------------------------------------------------

X = df[feature_columns].copy()
y = df[TARGET_COLUMN].copy()

print("\n" + "=" * 70)
print("X / y CHECK")
print("=" * 70)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget proportions:")
print(y.value_counts(normalize=True))

# ------------------------------------------------------------
# 6. Numeric check
# ------------------------------------------------------------

non_numeric = X.select_dtypes(
    exclude=[np.number]
).columns.tolist()

print("\n" + "=" * 70)
print("NUMERIC FEATURE CHECK")
print("=" * 70)

print("Non-numeric features:", non_numeric)

if non_numeric:
    raise ValueError(
        f"Non-numeric features found: {non_numeric}"
    )

print("✓ All 27 features are numeric")

# ------------------------------------------------------------
# 7. Final verification
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 2 COMPLETED")
print("=" * 70)

print("✓ Dataset loaded")
print("✓ 27 features verified")
print("✓ Target verified")
print("✓ All features numeric")

print("\nDataset is ready for hybrid model training.")

In [ ]:
# ============================================================
# HYBRID DL MODEL - STEP 3
# Train/Test Split + Scaling
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

print("=" * 70)
print("STEP 3 - TRAIN/TEST SPLIT + SCALING")
print("=" * 70)

# ------------------------------------------------------------
# 1. Train/Test Split
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTrain shape:", X_train.shape)
print("Test shape :", X_test.shape)

print("\nTrain target distribution:")
print(y_train.value_counts())

print("\nTest target distribution:")
print(y_test.value_counts())

# ------------------------------------------------------------
# 2. Convert to NumPy
# ------------------------------------------------------------

X_train_np = X_train.values.astype(np.float32)
X_test_np = X_test.values.astype(np.float32)

y_train_np = y_train.values.astype(np.int64)
y_test_np = y_test.values.astype(np.int64)

# ------------------------------------------------------------
# 3. Fit scaler ONLY on training data
# ------------------------------------------------------------

scaler_hybrid = StandardScaler()

X_train_scaled = scaler_hybrid.fit_transform(X_train_np)
X_test_scaled = scaler_hybrid.transform(X_test_np)

X_train_scaled = X_train_scaled.astype(np.float32)
X_test_scaled = X_test_scaled.astype(np.float32)

# ------------------------------------------------------------
# 4. Verify scaling
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SCALING CHECK")
print("=" * 70)

print("Train mean (first 5 features):")
print(np.mean(X_train_scaled[:, :5], axis=0))

print("\nTrain std (first 5 features):")
print(np.std(X_train_scaled[:, :5], axis=0))

print("\nTest mean (first 5 features):")
print(np.mean(X_test_scaled[:, :5], axis=0))

# ------------------------------------------------------------
# 5. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 3 COMPLETED")
print("=" * 70)

print("✓ 80:20 stratified train/test split")
print("✓ Random state = 42")
print("✓ Scaler fitted only on training data")
print("✓ Test data transformed using training scaler")
print("✓ No scaling leakage")

print("\nReady for STEP 4:")
print("Build the MLP + Tabular ResNet hybrid architecture.")

In [ ]:
# ============================================================
# HYBRID DL MODEL - STEP 4
# MLP + Tabular ResNet Architecture
# ============================================================

import torch
import torch.nn as nn

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print("STEP 4 - HYBRID MODEL ARCHITECTURE")
print("=" * 70)

print("Device:", device)

# ------------------------------------------------------------
# Residual Block
# ------------------------------------------------------------

class ResidualBlock(nn.Module):

    def __init__(self, dim, dropout=0.20):
        super().__init__()

        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim)
        )

        self.activation = nn.ReLU()

    def forward(self, x):
        residual = x

        out = self.block(x)

        out = out + residual

        return self.activation(out)


# ------------------------------------------------------------
# Hybrid MLP + ResNet
# ------------------------------------------------------------

class HybridMLPResNet(nn.Module):

    def __init__(
        self,
        input_dim=27,
        hidden_dim=128,
        num_res_blocks=3,
        dropout=0.20
    ):
        super().__init__()

        # -------------------------
        # Shared input projection
        # -------------------------

        self.input_layer = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # -------------------------
        # MLP branch
        # -------------------------

        self.mlp_branch = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # -------------------------
        # ResNet branch
        # -------------------------

        self.resnet_branch = nn.Sequential(
            ResidualBlock(hidden_dim, dropout),
            ResidualBlock(hidden_dim, dropout),
            ResidualBlock(hidden_dim, dropout)
        )

        self.resnet_projection = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # -------------------------
        # Fusion layer
        # -------------------------

        self.classifier = nn.Sequential(
            nn.Linear(64 + 64, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(64, 1)
        )

    def forward(self, x):

        # Shared representation
        shared = self.input_layer(x)

        # MLP branch
        mlp_features = self.mlp_branch(shared)

        # ResNet branch
        resnet_features = self.resnet_branch(shared)
        resnet_features = self.resnet_projection(resnet_features)

        # Feature fusion
        fused = torch.cat(
            [mlp_features, resnet_features],
            dim=1
        )

        # Final prediction
        output = self.classifier(fused)

        return output


# ------------------------------------------------------------
# Create model
# ------------------------------------------------------------

hybrid_model = HybridMLPResNet(
    input_dim=X_train_scaled.shape[1],
    hidden_dim=128,
    num_res_blocks=3,
    dropout=0.20
).to(device)

# ------------------------------------------------------------
# Model information
# ------------------------------------------------------------

print("\nModel:")
print(hybrid_model)

total_params = sum(
    p.numel() for p in hybrid_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in hybrid_model.parameters()
    if p.requires_grad
)

print("\n" + "=" * 70)
print("MODEL SUMMARY")
print("=" * 70)

print("Input features       :", X_train_scaled.shape[1])
print("Hidden dimension     :", 128)
print("Residual blocks      :", 3)
print("Dropout              :", 0.20)
print("Total parameters     :", f"{total_params:,}")
print("Trainable parameters :", f"{trainable_params:,}")

# ------------------------------------------------------------
# Test forward pass
# ------------------------------------------------------------

test_tensor = torch.tensor(
    X_train_scaled[:8],
    dtype=torch.float32
).to(device)

with torch.no_grad():
    test_output = hybrid_model(test_tensor)

print("\nForward pass:")
print("Input shape :", test_tensor.shape)
print("Output shape:", test_output.shape)

assert test_output.shape == (8, 1)

print("\n✓ Forward pass successful")
print("✓ MLP branch created")
print("✓ ResNet branch created")
print("✓ Feature fusion created")
print("✓ Binary classifier created")

print("\n" + "=" * 70)
print("STEP 4 COMPLETED")
print("=" * 70)

print("Ready for STEP 5:")
print("Training setup + class imbalance handling.")

In [ ]:
# ============================================================
# HYBRID DL MODEL - STEP 5
# Training Setup + Class Imbalance Handling
# ============================================================

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

print("=" * 70)
print("STEP 5 - TRAINING SETUP")
print("=" * 70)

# ------------------------------------------------------------
# 1. Convert data to PyTorch tensors
# ------------------------------------------------------------

X_train_tensor = torch.tensor(
    X_train_scaled,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train_np,
    dtype=torch.float32
).view(-1, 1)

X_test_tensor = torch.tensor(
    X_test_scaled,
    dtype=torch.float32
)

y_test_tensor = torch.tensor(
    y_test_np,
    dtype=torch.float32
).view(-1, 1)

print("X_train:", X_train_tensor.shape)
print("y_train:", y_train_tensor.shape)
print("X_test :", X_test_tensor.shape)
print("y_test :", y_test_tensor.shape)

# ------------------------------------------------------------
# 2. Calculate class weights
# ------------------------------------------------------------

class_counts = np.bincount(y_train_np)

negative_count = class_counts[0]
positive_count = class_counts[1]

total_count = len(y_train_np)

weight_negative = total_count / (2 * negative_count)
weight_positive = total_count / (2 * positive_count)

print("\n" + "=" * 70)
print("CLASS IMBALANCE")
print("=" * 70)

print("Class 0:", negative_count)
print("Class 1:", positive_count)

print("\nClass weight 0:", weight_negative)
print("Class weight 1:", weight_positive)

# ------------------------------------------------------------
# 3. Weighted BCE loss
# ------------------------------------------------------------

pos_weight = torch.tensor(
    [negative_count / positive_count],
    dtype=torch.float32
).to(device)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

print("\nPositive class weight:", pos_weight.item())

# ------------------------------------------------------------
# 4. DataLoader
# ------------------------------------------------------------

BATCH_SIZE = 256

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False
)

print("\n" + "=" * 70)
print("DATALOADER")
print("=" * 70)

print("Batch size:", BATCH_SIZE)
print("Training batches:", len(train_loader))
print("Testing batches :", len(test_loader))

# ------------------------------------------------------------
# 5. Optimizer
# ------------------------------------------------------------

LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4

optimizer = torch.optim.AdamW(
    hybrid_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# ------------------------------------------------------------
# 6. Learning-rate scheduler
# ------------------------------------------------------------

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)

print("\n" + "=" * 70)
print("OPTIMIZATION SETUP")
print("=" * 70)

print("Optimizer      : AdamW")
print("Learning rate  :", LEARNING_RATE)
print("Weight decay   :", WEIGHT_DECAY)
print("Scheduler      : ReduceLROnPlateau")
print("LR reduction   : 0.5")
print("Scheduler patience:", 3)

print("\n" + "=" * 70)
print("STEP 5 COMPLETED")
print("=" * 70)

print("✓ PyTorch tensors created")
print("✓ Class imbalance handled using weighted BCE")
print("✓ DataLoaders created")
print("✓ AdamW optimizer configured")
print("✓ Learning-rate scheduler configured")

print("\nReady for STEP 6:")
print("Hybrid model training with validation monitoring.")

In [ ]:
# ============================================================
# HYBRID DL MODEL - STEP 6
# Model Training + Validation Monitoring
# ============================================================

import copy
import time
import torch

print("=" * 70)
print("STEP 6 - HYBRID MODEL TRAINING")
print("=" * 70)

# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------

EPOCHS = 50
PATIENCE = 8

best_model_state = None
best_val_loss = float("inf")
best_epoch = 0
early_stop_counter = 0

history = {
    "train_loss": [],
    "val_loss": []
}

start_time = time.time()

# ------------------------------------------------------------
# Training loop
# ------------------------------------------------------------

for epoch in range(1, EPOCHS + 1):

    # ========================================================
    # TRAINING
    # ========================================================

    hybrid_model.train()

    train_loss_sum = 0.0
    train_samples = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = hybrid_model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        optimizer.step()

        batch_size = X_batch.size(0)

        train_loss_sum += loss.item() * batch_size
        train_samples += batch_size

    train_loss = train_loss_sum / train_samples

    # ========================================================
    # VALIDATION
    # ========================================================

    hybrid_model.eval()

    val_loss_sum = 0.0
    val_samples = 0

    with torch.no_grad():

        for X_batch, y_batch in test_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = hybrid_model(X_batch)

            loss = criterion(outputs, y_batch)

            batch_size = X_batch.size(0)

            val_loss_sum += loss.item() * batch_size
            val_samples += batch_size

    val_loss = val_loss_sum / val_samples

    # ========================================================
    # Scheduler
    # ========================================================

    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    # Save history
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    # ========================================================
    # Early stopping
    # ========================================================

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_epoch = epoch

        best_model_state = copy.deepcopy(
            hybrid_model.state_dict()
        )

        early_stop_counter = 0

        status = "✓ Best"

    else:

        early_stop_counter += 1

        status = (
            f"EarlyStop: {early_stop_counter}/{PATIENCE}"
        )

    # ========================================================
    # Print progress
    # ========================================================

    print(
        f"Epoch [{epoch:02d}/{EPOCHS}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"LR: {current_lr:.6f} | "
        f"{status}"
    )

    # Stop if no improvement
    if early_stop_counter >= PATIENCE:

        print("\nEarly stopping triggered.")
        break


# ------------------------------------------------------------
# Restore best model
# ------------------------------------------------------------

hybrid_model.load_state_dict(best_model_state)

training_time = time.time() - start_time

print("\n" + "=" * 70)
print("STEP 6 TRAINING COMPLETED")
print("=" * 70)

print(f"Best epoch        : {best_epoch}")
print(f"Best validation loss: {best_val_loss:.6f}")
print(f"Total epochs run  : {epoch}")
print(f"Training time     : {training_time / 60:.2f} minutes")

print("\n✓ Best model restored")
print("✓ Training history saved in memory")
print("✓ Ready for STEP 7: Final test evaluation")

In [ ]:
# ============================================================
# STEP 7 - FINAL TEST EVALUATION
# ============================================================

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print("\n" + "=" * 70)
print("STEP 7 - FINAL TEST EVALUATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Evaluation mode
# ------------------------------------------------------------

hybrid_model.eval()

all_probs = []
all_preds = []
all_targets = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # Model output
        outputs = hybrid_model(X_batch)

        # Convert logits to probability
        probs = torch.sigmoid(outputs)

        # Binary prediction
        preds = (probs >= 0.5).float()

        all_probs.extend(
            probs.cpu().numpy().reshape(-1)
        )

        all_preds.extend(
            preds.cpu().numpy().reshape(-1)
        )

        all_targets.extend(
            y_batch.cpu().numpy().reshape(-1)
        )

# ------------------------------------------------------------
# 2. Convert to NumPy arrays
# ------------------------------------------------------------

y_true = np.array(all_targets).astype(int)
y_pred = np.array(all_preds).astype(int)
y_prob = np.array(all_probs)

# ------------------------------------------------------------
# 3. Calculate evaluation metrics
# ------------------------------------------------------------

accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_true,
    y_prob
)

cm = confusion_matrix(
    y_true,
    y_pred
)

# ------------------------------------------------------------
# 4. Print results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL TEST RESULTS")
print("=" * 70)

print(f"Test samples : {len(y_true)}")
print(f"Accuracy     : {accuracy:.6f} ({accuracy * 100:.2f}%)")
print(f"Precision    : {precision:.6f} ({precision * 100:.2f}%)")
print(f"Recall       : {recall:.6f} ({recall * 100:.2f}%)")
print(f"F1-Score     : {f1:.6f} ({f1 * 100:.2f}%)")
print(f"ROC-AUC      : {roc_auc:.6f} ({roc_auc * 100:.2f}%)")

# ------------------------------------------------------------
# 5. Confusion Matrix
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

print("                 Predicted")
print("                 0       1")
print(f"Actual 0      {cm[0,0]:6d}  {cm[0,1]:6d}")
print(f"Actual 1      {cm[1,0]:6d}  {cm[1,1]:6d}")

# ------------------------------------------------------------
# 6. Classification Report
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        y_true,
        y_pred,
        target_names=["Class 0", "Class 1"],
        digits=4,
        zero_division=0
    )
)

print("\n" + "=" * 70)
print("STEP 7 COMPLETED")
print("=" * 70)

print("✓ Best trained model evaluated")
print("✓ Test predictions generated")
print("✓ Accuracy calculated")
print("✓ Precision calculated")
print("✓ Recall calculated")
print("✓ F1-score calculated")
print("✓ ROC-AUC calculated")
print("✓ Confusion matrix generated")

print("\nReady for STEP 8:")
print("ROC curve + Precision-Recall curve + Training curves")

## STEP 19 — Cross-Validation for All Final Models


In [ ]:
# ============================================================
# 10-FOLD STRATIFIED CROSS-VALIDATION
# ALL FINAL MODELS
# ============================================================

import numpy as np
import pandas as pd
import os
import pickle
import time

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# XGBoost
from xgboost import XGBClassifier

# ------------------------------------------------------------
# 1. Check required data
# ------------------------------------------------------------

required_vars = ["X_train", "y_train"]

missing = [v for v in required_vars if v not in globals()]

if missing:
    print("Missing variables:", missing)
    print("\nTrying to reconstruct X_train from scaled data...")

    # If raw X_train is unavailable, use the already scaled data.
    if "X_train_scaled" in globals():
        X_train_cv = X_train_scaled
    else:
        raise NameError(
            "X_train or X_train_scaled is not available. "
            "Run the dataset preparation cell first."
        )
else:
    X_train_cv = X_train

y_train_cv = y_train

print("=" * 70)
print("10-FOLD STRATIFIED CROSS-VALIDATION")
print("=" * 70)

print("Training data shape:", X_train_cv.shape)
print("Target shape       :", y_train_cv.shape)

# ------------------------------------------------------------
# 2. Stratified 10-Fold CV
# ------------------------------------------------------------

skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

# ------------------------------------------------------------
# 3. Define models
# ------------------------------------------------------------

models = {

    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]),

    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            probability=True,
            random_state=42
        ))
    ]),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ),

    "Class-Weighted Random Forest": RandomForestClassifier(
        n_estimators=152,
        max_depth=25,
        max_features="log2",
        min_samples_split=3,
        min_samples_leaf=1,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    ),

    "MLP": Pipeline([
        ("scaler", StandardScaler()),
        ("model", __import__(
            "sklearn.neural_network",
            fromlist=["MLPClassifier"]
        ).MLPClassifier(
            hidden_layer_sizes=(128, 64),
            activation="relu",
            solver="adam",
            alpha=0.0001,
            learning_rate_init=0.001,
            batch_size=32,
            max_iter=100,
            early_stopping=True,
            random_state=42
        ))
    ])
}

# ------------------------------------------------------------
# 4. Storage
# ------------------------------------------------------------

cv_results = []
fold_results = []

start_time = time.time()

# ------------------------------------------------------------
# 5. Run 10-Fold CV
# ------------------------------------------------------------

for model_name, model in models.items():

    print("\n" + "=" * 70)
    print(f"MODEL: {model_name}")
    print("=" * 70)

    model_scores = {
        "Accuracy": [],
        "Precision": [],
        "Recall": [],
        "F1-score": [],
        "ROC-AUC": []
    }

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(X_train_cv, y_train_cv), start=1
    ):

        X_fold_train = X_train_cv.iloc[train_idx] \
            if hasattr(X_train_cv, "iloc") else X_train_cv[train_idx]

        X_fold_val = X_train_cv.iloc[val_idx] \
            if hasattr(X_train_cv, "iloc") else X_train_cv[val_idx]

        y_fold_train = y_train_cv.iloc[train_idx] \
            if hasattr(y_train_cv, "iloc") else y_train_cv[train_idx]

        y_fold_val = y_train_cv.iloc[val_idx] \
            if hasattr(y_train_cv, "iloc") else y_train_cv[val_idx]

        # Train
        model.fit(X_fold_train, y_fold_train)

        # Prediction
        y_val_pred = model.predict(X_fold_val)

        # Probability / decision score
        if hasattr(model, "predict_proba"):
            y_val_prob = model.predict_proba(X_fold_val)[:, 1]

        elif hasattr(model, "decision_function"):
            y_val_prob = model.decision_function(X_fold_val)

        else:
            y_val_prob = y_val_pred

        # Metrics
        acc = accuracy_score(y_fold_val, y_val_pred)
        prec = precision_score(
            y_fold_val,
            y_val_pred,
            zero_division=0
        )
        rec = recall_score(
            y_fold_val,
            y_val_pred,
            zero_division=0
        )
        f1 = f1_score(
            y_fold_val,
            y_val_pred,
            zero_division=0
        )
        auc = roc_auc_score(
            y_fold_val,
            y_val_prob
        )

        model_scores["Accuracy"].append(acc)
        model_scores["Precision"].append(prec)
        model_scores["Recall"].append(rec)
        model_scores["F1-score"].append(f1)
        model_scores["ROC-AUC"].append(auc)

        fold_results.append({
            "Model": model_name,
            "Fold": fold,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1-score": f1,
            "ROC-AUC": auc
        })

        print(
            f"Fold {fold:2d} | "
            f"Accuracy={acc:.4f} | "
            f"F1={f1:.4f} | "
            f"ROC-AUC={auc:.4f}"
        )

    # --------------------------------------------------------
    # Mean ± Standard Deviation
    # --------------------------------------------------------

    summary = {
        "Model": model_name,

        "Accuracy_Mean":
            np.mean(model_scores["Accuracy"]),

        "Accuracy_Std":
            np.std(model_scores["Accuracy"], ddof=1),

        "Precision_Mean":
            np.mean(model_scores["Precision"]),

        "Precision_Std":
            np.std(model_scores["Precision"], ddof=1),

        "Recall_Mean":
            np.mean(model_scores["Recall"]),

        "Recall_Std":
            np.std(model_scores["Recall"], ddof=1),

        "F1_Mean":
            np.mean(model_scores["F1-score"]),

        "F1_Std":
            np.std(model_scores["F1-score"], ddof=1),

        "ROC_AUC_Mean":
            np.mean(model_scores["ROC-AUC"]),

        "ROC_AUC_Std":
            np.std(model_scores["ROC-AUC"], ddof=1)
    }

    cv_results.append(summary)

    print("\nMean ± Std:")
    print(
        f"Accuracy : "
        f"{summary['Accuracy_Mean']:.6f} ± "
        f"{summary['Accuracy_Std']:.6f}"
    )

    print(
        f"Precision: "
        f"{summary['Precision_Mean']:.6f} ± "
        f"{summary['Precision_Std']:.6f}"
    )

    print(
        f"Recall   : "
        f"{summary['Recall_Mean']:.6f} ± "
        f"{summary['Recall_Std']:.6f}"
    )

    print(
        f"F1-score : "
        f"{summary['F1_Mean']:.6f} ± "
        f"{summary['F1_Std']:.6f}"
    )

    print(
        f"ROC-AUC  : "
        f"{summary['ROC_AUC_Mean']:.6f} ± "
        f"{summary['ROC_AUC_Std']:.6f}"
    )

# ------------------------------------------------------------
# 6. Convert to DataFrames
# ------------------------------------------------------------

cv_summary_df = pd.DataFrame(cv_results)
cv_fold_df = pd.DataFrame(fold_results)

# Sort by F1-score
cv_summary_df = cv_summary_df.sort_values(
    "F1_Mean",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 70)
print("===== FINAL 10-FOLD CROSS-VALIDATION SUMMARY =====")
print("=" * 70)

display(cv_summary_df)

# ------------------------------------------------------------
# 7. Save results
# ------------------------------------------------------------

saved_dir = MODEL_DIR

os.makedirs(saved_dir, exist_ok=True)

cv_summary_df.to_csv(
    f"{saved_dir}/all_models_cv_summary.csv",
    index=False
)

cv_fold_df.to_csv(
    f"{saved_dir}/all_models_cv_fold_results.csv",
    index=False
)

print("\n✓ CV summary saved:")
print("  all_models_cv_summary.csv")

print("\n✓ Fold-wise results saved:")
print("  all_models_cv_fold_results.csv")

print(
    f"\nTotal CV time: "
    f"{(time.time() - start_time)/60:.2f} minutes"
)

print("\n" + "=" * 70)
print("10-FOLD CROSS-VALIDATION COMPLETED")
print("=" * 70)

## STEP 20 — Statistical Significance Testing
Paired t-test and Wilcoxon Signed-Rank Test are used for fold-wise comparisons.


In [ ]:
# ============================================================
# STATISTICAL SIGNIFICANCE TESTING
# Paired t-test + Wilcoxon Signed-Rank Test
# ============================================================

import pandas as pd
import numpy as np
import os

from scipy.stats import ttest_rel, wilcoxon

# ------------------------------------------------------------
# 1. Load saved fold-wise CV results
# ------------------------------------------------------------

saved_dir = MODEL_DIR

cv_file = f"{saved_dir}/all_models_cv_fold_results.csv"

cv_df = pd.read_csv(cv_file)

print("=" * 70)
print("STATISTICAL SIGNIFICANCE TESTING")
print("=" * 70)

print("\nLoaded CV results:")
print(cv_df.shape)

print("\nModels:")
print(cv_df["Model"].unique())

# ------------------------------------------------------------
# 2. Metrics to test
# ------------------------------------------------------------

metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "ROC-AUC"
]

# ------------------------------------------------------------
# 3. Model pairs
# ------------------------------------------------------------

model_pairs = [
    ("Class-Weighted Random Forest", "MLP"),
    ("Class-Weighted Random Forest", "XGBoost"),
    ("MLP", "XGBoost"),
    ("MLP", "SVM"),
    ("MLP", "Decision Tree"),
    ("MLP", "Logistic Regression")
]

# ------------------------------------------------------------
# 4. Statistical tests
# ------------------------------------------------------------

results = []

for model_a, model_b in model_pairs:

    print("\n" + "=" * 70)
    print(f"{model_a}  VS  {model_b}")
    print("=" * 70)

    for metric in metrics:

        a = (
            cv_df[cv_df["Model"] == model_a]
            .sort_values("Fold")[metric]
            .values
        )

        b = (
            cv_df[cv_df["Model"] == model_b]
            .sort_values("Fold")[metric]
            .values
        )

        # ----------------------------------------------------
        # Paired t-test
        # ----------------------------------------------------

        t_stat, t_p = ttest_rel(a, b)

        # ----------------------------------------------------
        # Wilcoxon signed-rank test
        # ----------------------------------------------------

        try:
            w_stat, w_p = wilcoxon(
                a,
                b,
                alternative="two-sided"
            )
        except ValueError:
            w_stat = np.nan
            w_p = np.nan

        # Mean difference
        mean_a = np.mean(a)
        mean_b = np.mean(b)
        mean_difference = mean_a - mean_b

        results.append({
            "Model_A": model_a,
            "Model_B": model_b,
            "Metric": metric,

            "Mean_A": mean_a,
            "Mean_B": mean_b,
            "Mean_Difference": mean_difference,

            "Paired_t_statistic": t_stat,
            "Paired_t_p_value": t_p,

            "Wilcoxon_statistic": w_stat,
            "Wilcoxon_p_value": w_p
        })

        significance = (
            "Significant"
            if w_p < 0.05
            else "Not Significant"
        )

        print(
            f"{metric:10s} | "
            f"Mean Diff = {mean_difference:+.6f} | "
            f"t-test p = {t_p:.6f} | "
            f"Wilcoxon p = {w_p:.6f} | "
            f"{significance}"
        )

# ------------------------------------------------------------
# 5. Create results DataFrame
# ------------------------------------------------------------

stats_df = pd.DataFrame(results)

# ------------------------------------------------------------
# 6. Add significance labels
# ------------------------------------------------------------

stats_df["Significant_0.05"] = (
    stats_df["Wilcoxon_p_value"] < 0.05
)

stats_df["Significance"] = np.where(
    stats_df["Wilcoxon_p_value"] < 0.05,
    "Significant",
    "Not Significant"
)

# ------------------------------------------------------------
# 7. Display complete results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("===== STATISTICAL TESTING SUMMARY =====")
print("=" * 70)

display(stats_df)

# ------------------------------------------------------------
# 8. Save results
# ------------------------------------------------------------

stats_file = f"{saved_dir}/statistical_significance_tests.csv"

stats_df.to_csv(
    stats_file,
    index=False
)

print("\n✓ Statistical testing results saved:")
print(stats_file)

# ------------------------------------------------------------
# 9. Compact paper-ready table
# ------------------------------------------------------------

paper_stats = stats_df[
    [
        "Model_A",
        "Model_B",
        "Metric",
        "Mean_Difference",
        "Paired_t_p_value",
        "Wilcoxon_p_value",
        "Significance"
    ]
].copy()

paper_stats["Mean_Difference"] = paper_stats[
    "Mean_Difference"
].round(6)

paper_stats["Paired_t_p_value"] = paper_stats[
    "Paired_t_p_value"
].round(6)

paper_stats["Wilcoxon_p_value"] = paper_stats[
    "Wilcoxon_p_value"
].round(6)

print("\n" + "=" * 70)
print("===== PAPER-READY STATISTICAL RESULTS =====")
print("=" * 70)

display(paper_stats)

paper_stats.to_csv(
    f"{saved_dir}/paper_statistical_results.csv",
    index=False
)

print("\n✓ Paper-ready table saved.")

In [ ]:
import os
import pickle
import pandas as pd

saved_dir = MODEL_DIR
project_dir = PROJECT_DIR

# Load final 27 feature names
with open(f"{saved_dir}/feature_names.pkl", "rb") as f:
    feature_names = pickle.load(f)

print("Number of final features:", len(feature_names))
print("\nFinal features:")
for i, feature in enumerate(feature_names, 1):
    print(f"{i}. {feature}")

# Save final 27 feature list
feature_file = f"{project_dir}/final_27_features.txt"

with open(feature_file, "w") as f:
    f.write("FINAL 27 SELECTED FEATURES\n")
    f.write("=" * 40 + "\n\n")

    for i, feature in enumerate(feature_names, 1):
        f.write(f"{i}. {feature}\n")

print("\n✓ final_27_features.txt saved")
print(feature_file)

In [ ]:
import os
import pandas as pd
import pickle

project_dir = PROJECT_DIR
saved_dir = f"{project_dir}/saved_state"

# Load 27 selected features
with open(f"{saved_dir}/feature_names.pkl", "rb") as f:
    features_27 = pickle.load(f)

# Try to identify the original 47 features
with open(f"{saved_dir}/selected_47_features.pkl", "rb") as f:
    features_47 = pickle.load(f)

print("47-feature count:", len(features_47))
print("27-feature count:", len(features_27))

# Create comparison
comparison = pd.DataFrame({
    "Feature": features_47,
    "Selected_in_27": [
        feature in features_27 for feature in features_47
    ]
})

comparison["Status"] = comparison["Selected_in_27"].map({
    True: "Selected",
    False: "Removed"
})

# Save
comparison_file = f"{project_dir}/feature_comparison_47_vs_27.csv"
comparison.to_csv(comparison_file, index=False)

print("\n" + "=" * 60)
print("47 VS 27 FEATURE COMPARISON")
print("=" * 60)

print(comparison)

print("\n✓ feature_comparison_47_vs_27.csv saved")
print(comparison_file)

In [ ]:
import pandas as pd

stats_path = f"{saved_dir}/paper_statistical_results.csv"

stats_df = pd.read_csv(stats_path)

print("=" * 60)
print("STATISTICAL RESULTS")
print("=" * 60)

display(stats_df)

## STEP 21 — SHAP Global Feature Importance


In [ ]:
# ============================================================
# L. EXPLAINABLE AI - SHAP GLOBAL FEATURE IMPORTANCE
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Install SHAP if needed
!pip -q install shap

import shap

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier


# ============================================================
# 1. Paths
# ============================================================

project_dir = PROJECT_DIR
results_dir = f"{project_dir}/results"

os.makedirs(results_dir, exist_ok=True)


# ============================================================
# 2. Load final 27-feature dataset
# ============================================================

data_path = f"{project_dir}/phishing_27_features.csv"

df = pd.read_csv(data_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)


# ============================================================
# 3. Prepare X and y
# ============================================================

X = df.drop(columns=["phishing"])
y = df["phishing"]

feature_names = X.columns.tolist()

print("\nNumber of features:", len(feature_names))


# ============================================================
# 4. Same 80:20 stratified split used for final MLP
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)


# ============================================================
# 5. Standardization
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# ============================================================
# 6. Recreate FINAL MLP
#    Same configuration selected earlier
# ============================================================

final_mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    batch_size=32,
    learning_rate_init=0.001,
    max_iter=100,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    random_state=42
)

print("\nTraining final MLP for SHAP analysis...")

final_mlp.fit(
    X_train_scaled,
    y_train
)

print("MLP training completed!")
print("Iterations:", final_mlp.n_iter_)


# ============================================================
# 7. Select background data for SHAP
# ============================================================

# Small background sample keeps SHAP computationally manageable
background_size = min(50, len(X_train_scaled))

background = shap.sample(
    X_train_scaled,
    background_size,
    random_state=42
)

print("\nSHAP background samples:", len(background))


# ============================================================
# 8. Select test samples to explain
# ============================================================

explain_size = min(300, len(X_test_scaled))

X_explain = X_test_scaled[:explain_size]

print("Samples selected for SHAP explanation:", len(X_explain))


# ============================================================
# 9. Create SHAP explainer
# ============================================================

print("\nCreating SHAP explainer...")
print("This may take some time...")

explainer = shap.Explainer(
    final_mlp.predict_proba,
    background,
    feature_names=feature_names
)


# ============================================================
# 10. Calculate SHAP values
# ============================================================

print("\nCalculating SHAP values...")

shap_values = explainer(
    X_explain,
    max_evals=300
)

print("SHAP calculation completed!")


# ============================================================
# 11. Handle binary-class output
# ============================================================

# We explain the phishing class (class = 1)
if shap_values.values.ndim == 3:
    phishing_shap_values = shap_values.values[:, :, 1]
else:
    phishing_shap_values = shap_values.values


# ============================================================
# 12. Calculate global feature importance
# ============================================================

mean_abs_shap = np.mean(
    np.abs(phishing_shap_values),
    axis=0
)

shap_importance = pd.DataFrame({
    "Feature": feature_names,
    "Mean_Absolute_SHAP": mean_abs_shap
})

shap_importance = shap_importance.sort_values(
    "Mean_Absolute_SHAP",
    ascending=False
).reset_index(drop=True)


# ============================================================
# 13. Display ranking
# ============================================================

print("\n===== SHAP GLOBAL FEATURE IMPORTANCE =====")

print(
    shap_importance.to_string(index=False)
)


# ============================================================
# 14. Save SHAP importance table
# ============================================================

shap_importance.to_csv(
    f"{results_dir}/shap_global_feature_importance.csv",
    index=False
)

print("\nSHAP feature importance saved successfully!")


# ============================================================
# 15. SHAP global bar plot
# ============================================================

plt.figure(figsize=(10, 8))

shap.summary_plot(
    phishing_shap_values,
    X_explain,
    feature_names=feature_names,
    plot_type="bar",
    show=False
)

plt.title(
    "SHAP Global Feature Importance for Phishing Detection"
)

plt.tight_layout()

shap_bar_path = (
    f"{results_dir}/shap_global_feature_importance.png"
)

plt.savefig(
    shap_bar_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nSHAP bar plot saved:")
print(shap_bar_path)


# ============================================================
# 16. SHAP summary / beeswarm plot
# ============================================================

plt.figure(figsize=(10, 8))

shap.summary_plot(
    phishing_shap_values,
    X_explain,
    feature_names=feature_names,
    show=False
)

plt.title(
    "SHAP Summary Plot for the Proposed MLP"
)

plt.tight_layout()

shap_summary_path = (
    f"{results_dir}/shap_summary_plot.png"
)

plt.savefig(
    shap_summary_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nSHAP summary plot saved:")
print(shap_summary_path)

print("\n===== SHAP ANALYSIS COMPLETED =====")

In [ ]:
import os
import joblib
import pickle
import numpy as np
import pandas as pd

# ============================================
# 1. Create saved_state folder in Google Drive
# ============================================

save_dir = MODEL_DIR
os.makedirs(save_dir, exist_ok=True)

print("Saving to:", save_dir)


# ============================================
# 2. Save trained MLP models
# ============================================

joblib.dump(final_mlp, os.path.join(save_dir, "final_mlp.pkl"))
joblib.dump(mlp_47, os.path.join(save_dir, "mlp_47.pkl"))

print("✓ Models saved")


# ============================================
# 3. Save scalers
# ============================================

joblib.dump(scaler, os.path.join(save_dir, "scaler.pkl"))
joblib.dump(scaler_47, os.path.join(save_dir, "scaler_47.pkl"))

print("✓ Scalers saved")


# ============================================
# 4. Save feature information
# ============================================

joblib.dump(feature_columns, os.path.join(save_dir, "feature_columns.pkl"))
joblib.dump(feature_names, os.path.join(save_dir, "feature_names.pkl"))
joblib.dump(selected_47_features, os.path.join(save_dir, "selected_47_features.pkl"))

print("✓ Feature information saved")


# ============================================
# 5. Save SHAP results
# ============================================

joblib.dump(phishing_shap_values,
            os.path.join(save_dir, "phishing_shap_values.pkl"))

joblib.dump(mean_abs_shap,
            os.path.join(save_dir, "mean_abs_shap.pkl"))

joblib.dump(shap_importance,
            os.path.join(save_dir, "shap_importance.pkl"))

# Save SHAP Explanation object if available
try:
    joblib.dump(shap_values,
                os.path.join(save_dir, "shap_values.pkl"))
    print("✓ SHAP values saved")
except Exception as e:
    print("⚠ SHAP Explanation could not be saved:", e)


# ============================================
# 6. Save important result DataFrames
# ============================================

dataframes_to_save = {
    "final_mlp_results": final_mlp_results,
    "mlp_results_df": mlp_results_df,
    "mlp_cv_summary": mlp_cv_summary,
    "mlp_confusion_matrix": mlp_confusion_matrix
}

for name, dataframe in dataframes_to_save.items():
    dataframe.to_csv(
        os.path.join(save_dir, f"{name}.csv"),
        index=False
    )

print("✓ Result DataFrames saved")


# ============================================
# 7. Save prediction/results arrays
# ============================================

arrays_to_save = {
    "y_pred": y_pred,
    "y_pred_47": y_pred_47,
    "y_prob": y_prob,
    "y_prob_47": y_prob_47,
    "y_test": y_test.to_numpy(),
    "y_test_47": y_test_47.to_numpy(),
    "y_train": y_train.to_numpy(),
    "y_train_47": y_train_47.to_numpy(),
    "scores": scores
}

for name, array in arrays_to_save.items():
    np.save(
        os.path.join(save_dir, f"{name}.npy"),
        array
    )

print("✓ Prediction arrays saved")


# ============================================
# 8. Save important metrics
# ============================================

metrics = {
    "precision": precision,
    "precision_47": precision_47,
    "recall": recall,
    "recall_47": recall_47,
    "roc_auc": float(roc_auc),
    "roc_auc_47": float(roc_auc_47)
}

pd.DataFrame([metrics]).to_csv(
    os.path.join(save_dir, "metrics.csv"),
    index=False
)

print("✓ Metrics saved")


# ============================================
# 9. Save a simple information file
# ============================================

with open(os.path.join(save_dir, "README.txt"), "w") as f:
    f.write("Saved state of Phishing Detection Project\n")
    f.write("Important models, scalers, features, SHAP results and evaluation results are stored here.\n")

print("✓ README saved")


# ============================================
# 10. Show everything that was saved
# ============================================

print("\n" + "="*50)
print("        SAVE COMPLETED SUCCESSFULLY")
print("="*50)

saved_files = os.listdir(save_dir)

for file in sorted(saved_files):
    print("✓", file)

print("\nTotal files saved:", len(saved_files))
print("Location:", save_dir)

In [ ]:
# ============================================================
# RESTORE SAVED MLP + SHAP STATE
# ============================================================

import os
import pickle
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Project paths
# ------------------------------------------------------------

project_dir = PROJECT_DIR
saved_dir = f"{project_dir}/saved_state"
results_dir = f"{project_dir}/results"

print("Saved state exists:", os.path.exists(saved_dir))

# ------------------------------------------------------------
# 2. Load saved model
# ------------------------------------------------------------

with open(f"{saved_dir}/final_mlp.pkl", "rb") as f:
    final_mlp = pickle.load(f)

# ------------------------------------------------------------
# 3. Load scaler
# ------------------------------------------------------------

with open(f"{saved_dir}/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

# ------------------------------------------------------------
# 4. Load feature information
# ------------------------------------------------------------

with open(f"{saved_dir}/feature_names.pkl", "rb") as f:
    feature_names = pickle.load(f)

# ------------------------------------------------------------
# 5. Load SHAP values
# ------------------------------------------------------------

with open(f"{saved_dir}/phishing_shap_values.pkl", "rb") as f:
    phishing_shap_values = pickle.load(f)

with open(f"{saved_dir}/shap_values.pkl", "rb") as f:
    shap_values_saved = pickle.load(f)

with open(f"{saved_dir}/shap_importance.pkl", "rb") as f:
    shap_importance = pickle.load(f)

with open(f"{saved_dir}/mean_abs_shap.pkl", "rb") as f:
    mean_abs_shap = pickle.load(f)

# ------------------------------------------------------------
# 6. Load predictions and test labels
# ------------------------------------------------------------

y_pred = np.load(f"{saved_dir}/y_pred.npy")
y_prob = np.load(f"{saved_dir}/y_prob.npy")
y_test = np.load(f"{saved_dir}/y_test.npy")

# ------------------------------------------------------------
# 7. Display restored information
# ------------------------------------------------------------

print("\n" + "="*55)
print("SAVED STATE RESTORED SUCCESSFULLY")
print("="*55)

print("Model:", type(final_mlp).__name__)
print("MLP architecture:", final_mlp.hidden_layer_sizes)
print("Number of features:", len(feature_names))
print("Test samples:", len(y_test))
print("SHAP samples:", len(phishing_shap_values))

print("\nFeatures:")
for i, feature in enumerate(feature_names, 1):
    print(f"{i}. {feature}")

print("\nSaved files restored successfully!")

In [ ]:
import os
import pickle
import numpy as np

project_dir = PROJECT_DIR
saved_dir = f"{project_dir}/saved_state"

# Load final MLP
with open(f"{saved_dir}/final_mlp.pkl", "rb") as f:
    final_mlp = pickle.load(f)

# Load scaler
with open(f"{saved_dir}/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

# Load feature names
with open(f"{saved_dir}/feature_names.pkl", "rb") as f:
    feature_names = pickle.load(f)

# Load SHAP values
with open(f"{saved_dir}/phishing_shap_values.pkl", "rb") as f:
    phishing_shap_values = pickle.load(f)

with open(f"{saved_dir}/shap_importance.pkl", "rb") as f:
    shap_importance = pickle.load(f)

# Load predictions
y_pred = np.load(f"{saved_dir}/y_pred.npy")
y_prob = np.load(f"{saved_dir}/y_prob.npy")
y_test = np.load(f"{saved_dir}/y_test.npy")

print("=" * 55)
print("SAVED STATE RESTORED SUCCESSFULLY")
print("=" * 55)

print("MLP architecture:", final_mlp.hidden_layer_sizes)
print("Number of features:", len(feature_names))
print("Test samples:", len(y_test))
print("SHAP samples:", len(phishing_shap_values))

print("\nAll required variables restored successfully! ✓")

In [ ]:
import os
import joblib
import numpy as np

saved_dir = MODEL_DIR

# ============================================================
# RESTORE SAVED STATE
# ============================================================

final_mlp = joblib.load(
    f"{saved_dir}/final_mlp.pkl"
)

scaler = joblib.load(
    f"{saved_dir}/scaler.pkl"
)

feature_names = joblib.load(
    f"{saved_dir}/feature_names.pkl"
)

phishing_shap_values = joblib.load(
    f"{saved_dir}/phishing_shap_values.pkl"
)

shap_values_saved = joblib.load(
    f"{saved_dir}/shap_values.pkl"
)

shap_importance = joblib.load(
    f"{saved_dir}/shap_importance.pkl"
)

mean_abs_shap = joblib.load(
    f"{saved_dir}/mean_abs_shap.pkl"
)

# ============================================================
# Load NumPy arrays
# ============================================================

y_pred = np.load(
    f"{saved_dir}/y_pred.npy"
)

y_prob = np.load(
    f"{saved_dir}/y_prob.npy"
)

y_test = np.load(
    f"{saved_dir}/y_test.npy"
)

# ============================================================
# Verify
# ============================================================

print("=" * 60)
print("SAVED STATE RESTORED SUCCESSFULLY")
print("=" * 60)

print("MLP type:", type(final_mlp))
print("MLP architecture:", final_mlp.hidden_layer_sizes)

print("Number of features:", len(feature_names))

print("Test samples:", len(y_test))

print(
    "SHAP shape:",
    np.array(phishing_shap_values).shape
)

print(
    "Prediction shape:",
    y_pred.shape
)

print(
    "Probability shape:",
    y_prob.shape
)

print("\n✓ Model restored")
print("✓ Scaler restored")
print("✓ Feature names restored")
print("✓ SHAP values restored")
print("✓ SHAP importance restored")
print("✓ Predictions restored")
print("✓ Test labels restored")

## STEP 22 — SHAP Local Explanation


In [ ]:
# ============================================================
# SHAP INDIVIDUAL PREDICTION ANALYSIS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# SHAP values were calculated for the first 300 test samples
# ------------------------------------------------------------

n_shap_samples = len(phishing_shap_values)

y_test_shap = y_test[:n_shap_samples]
y_pred_shap = y_pred[:n_shap_samples]
y_prob_shap = y_prob[:n_shap_samples]

# ------------------------------------------------------------
# Find correctly predicted phishing samples
# ------------------------------------------------------------

correct_phishing = np.where(
    (y_test_shap == 1) &
    (y_pred_shap == 1)
)[0]

# ------------------------------------------------------------
# Find correctly predicted legitimate samples
# ------------------------------------------------------------

correct_legitimate = np.where(
    (y_test_shap == 0) &
    (y_pred_shap == 0)
)[0]

print("Correctly predicted phishing samples:", len(correct_phishing))
print("Correctly predicted legitimate samples:", len(correct_legitimate))

# ------------------------------------------------------------
# Select first correctly predicted sample from each class
# ------------------------------------------------------------

phishing_idx = correct_phishing[0]
legitimate_idx = correct_legitimate[0]

print("\nSelected phishing sample index:", phishing_idx)
print("Selected legitimate sample index:", legitimate_idx)

print("\nPhishing sample:")
print("Actual:", y_test_shap[phishing_idx])
print("Predicted:", y_pred_shap[phishing_idx])
print("Phishing probability:", y_prob_shap[phishing_idx])

print("\nLegitimate sample:")
print("Actual:", y_test_shap[legitimate_idx])
print("Predicted:", y_pred_shap[legitimate_idx])
print("Phishing probability:", y_prob_shap[legitimate_idx])


# ============================================================
# FUNCTION TO DISPLAY INDIVIDUAL SHAP CONTRIBUTIONS
# ============================================================

def individual_shap_analysis(index, sample_type):

    values = np.array(phishing_shap_values[index])

    explanation_df = pd.DataFrame({
        "Feature": feature_names,
        "SHAP_Value": values,
        "Absolute_SHAP": np.abs(values)
    })

    explanation_df = explanation_df.sort_values(
        "Absolute_SHAP",
        ascending=False
    ).reset_index(drop=True)

    print("\n" + "=" * 60)
    print(f"INDIVIDUAL SHAP ANALYSIS: {sample_type}")
    print("=" * 60)

    print(
        explanation_df[
            ["Feature", "SHAP_Value"]
        ].head(10).to_string(index=False)
    )

    # --------------------------------------------------------
    # Save complete explanation
    # --------------------------------------------------------

    safe_name = sample_type.lower().replace(" ", "_")

    explanation_df.to_csv(
        f"{results_dir}/shap_individual_{safe_name}.csv",
        index=False
    )

    # --------------------------------------------------------
    # Plot top 10 features
    # --------------------------------------------------------

    top10 = explanation_df.head(10).copy()

    top10 = top10.sort_values(
        "SHAP_Value"
    )

    plt.figure(figsize=(10, 6))

    plt.barh(
        top10["Feature"],
        top10["SHAP_Value"]
    )

    plt.axvline(
        0,
        linewidth=1
    )

    plt.xlabel("SHAP value")
    plt.ylabel("Feature")

    plt.title(
        f"Individual SHAP Explanation - {sample_type}"
    )

    plt.tight_layout()

    plot_path = (
        f"{results_dir}/shap_individual_{safe_name}.png"
    )

    plt.savefig(
        plot_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print(
        f"\nSaved CSV: shap_individual_{safe_name}.csv"
    )

    print(
        f"Saved plot: shap_individual_{safe_name}.png"
    )

    return explanation_df


# ============================================================
# Analyze phishing sample
# ============================================================

phishing_explanation = individual_shap_analysis(
    phishing_idx,
    "Correctly Predicted Phishing"
)


# ============================================================
# Analyze legitimate sample
# ============================================================

legitimate_explanation = individual_shap_analysis(
    legitimate_idx,
    "Correctly Predicted Legitimate"
)

## STEP 23 — LIME Local Explanation


In [ ]:
# ============================================================
# L. EXPLAINABLE AI - LIME INDIVIDUAL EXPLANATION
# ============================================================

!pip -q install lime

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lime.lime_tabular import LimeTabularExplainer


# ============================================================
# 1. Load original 27-feature dataset
# ============================================================

data_path = (
    "/content/gdrive/MyDrive/"
    "Phishing_Detection_Project/phishing_27_features.csv"
)

df = pd.read_csv(data_path)

X = df.drop(columns=["phishing"])
y = df["phishing"]

feature_names = X.columns.tolist()


# ============================================================
# 2. Recreate train/test split
#    Same split used previously
# ============================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test_original = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)


# ============================================================
# 3. Scale test data using SAVED scaler
# ============================================================

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)


# ============================================================
# 4. Create LIME explainer
# ============================================================

lime_explainer = LimeTabularExplainer(
    X_train_scaled,
    feature_names=feature_names,
    class_names=["Legitimate", "Phishing"],
    mode="classification",
    discretize_continuous=True,
    random_state=42
)

print("LIME explainer created successfully!")


# ============================================================
# 5. Use same SHAP sample indices
# ============================================================

phishing_index = phishing_idx
legitimate_index = legitimate_idx


# ============================================================
# 6. Function for LIME explanation
# ============================================================

def run_lime_explanation(index, sample_type):

    print("\n" + "=" * 60)
    print(f"LIME ANALYSIS: {sample_type}")
    print("=" * 60)

    instance = X_test_scaled[index]

    # Generate explanation
    explanation = lime_explainer.explain_instance(
        instance,
        final_mlp.predict_proba,
        num_features=10,
        num_samples=3000
    )

    # Get explanation for phishing class
    lime_list = explanation.as_list(
        label=1
    )

    lime_df = pd.DataFrame(
        lime_list,
        columns=["Feature_Condition", "LIME_Weight"]
    )

    print(lime_df.to_string(index=False))

    # --------------------------------------------------------
    # Save explanation
    # --------------------------------------------------------

    safe_name = (
        sample_type.lower()
        .replace(" ", "_")
    )

    csv_path = (
        f"{results_dir}/lime_individual_{safe_name}.csv"
    )

    lime_df.to_csv(
        csv_path,
        index=False
    )

    # --------------------------------------------------------
    # Save plot
    # --------------------------------------------------------

    plot_path = (
        f"{results_dir}/lime_individual_{safe_name}.png"
    )

    fig = explanation.as_pyplot_figure(
        label=1
    )

    fig.set_size_inches(10, 7)

    plt.title(
        f"LIME Explanation - {sample_type}"
    )

    plt.tight_layout()

    plt.savefig(
        plot_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print("\nSaved CSV:")
    print(csv_path)

    print("\nSaved plot:")
    print(plot_path)

    return explanation, lime_df


# ============================================================
# 7. Correctly predicted phishing
# ============================================================

phishing_lime_explanation, phishing_lime_df = (
    run_lime_explanation(
        phishing_index,
        "Correctly Predicted Phishing"
    )
)


# ============================================================
# 8. Correctly predicted legitimate
# ============================================================

legitimate_lime_explanation, legitimate_lime_df = (
    run_lime_explanation(
        legitimate_index,
        "Correctly Predicted Legitimate"
    )
)


print("\n" + "=" * 60)
print("LIME ANALYSIS COMPLETED")
print("=" * 60)

## End of Master Pipeline
The notebook follows the paper workflow: preprocessing → 47-feature selection → 27-feature reduction → ML/DL experiments → validation → statistics → SHAP/LIME.
